# CS781 Phase 4. CJPE data preparation and preflight

Court Judgment Prediction assigns one binary outcome to each case document.
I reused the Phase 1 byte BPE tokenizer and prepared one cache for the four
backbones. The preflight checked model construction, padding, gradients,
checkpoint reload and GPU memory before downstream training.

| Official split | Rows | Distinct original case IDs |
| --- | ---: | ---: |
| Train | 37,387 | 32,305 |
| Dev | 1,988 | 994 |
| Test | 1,517 | 1,517 |

The preparation assigns a unique internal row ID to every supplied row and
keeps the original case ID for submission export. The saved outputs record
the successful data checks and four passing model probes.

## 1. Inputs and settings

Use a Kaggle session with two T4 GPUs. For the official download, enable
Internet and grant the notebook access to the Kaggle secret `HF_TOKEN`.
To reproduce the saved run, set `HF_REVISION` to
`93b0059be756c6c461f3a225de25aabda7d5f84a`. Set `CJPE_DATA_INPUT` to the official data
folder or ZIP if you already have those files.

Attach [`prep_002_mixed16k`](../phase1/phase1_preparation_16k.ipynb)
with `tokenizer_state.pkl` and `cache_manifest.json`, plus the original
Phase 1 `best_model.pt` for each backbone.

| Backbone | Phase 1 run |
| --- | --- |
| decoder | [`decoder_004_mixed217m`](../phase1/phase1_decoder_training.ipynb) |
| encoder-decoder | [`encdec_002_mixed217m_stable`](../phase1/phase1_encoder_decoder_training.ipynb) |
| prefix | [`prefix_001_mixed217m`](../phase1/phase1_prefix_training.ipynb) |
| efficient | [`efficient_002_mixed217m_500m`](../phase1/phase1_efficient_500m_continuation.ipynb) |

Each run can be attached as an extracted folder or its backup ZIP.
Set `PREP_INPUT` and the entries in `PHASE1_INPUTS` to the mounted paths
when automatic discovery is ambiguous. Keep the same `RUN_ID` and working
directory to reuse completed token caches after an interruption.

In [1]:
from pathlib import Path
import json
import sys

RUN_ID = "phase4_cjpe_preparation_001"
PREP_INPUT = None
PHASE1_INPUTS = {
    "decoder": None,
    "encoder_decoder": None,
    "prefix": None,
    "efficient": None,
}
CJPE_DATA_INPUT = None       # Optional official Phase_4_CJPE folder/ZIP.
HF_REVISION = None           # Resolve and pin the current official commit.
LABEL_FIELD = None           # Detect explicit scalar 0/1 target fields.
RUN_MODEL_PROBES = True
RUN_RESOURCE_PROBES = True
SMOKE_STEPS = 12

ROOT = Path("/kaggle/working/cs781_phase4_prep") / RUN_ID
SRC = Path("/kaggle/working/cs781_phase4_prep_src")
ROOT.mkdir(parents=True, exist_ok=True)
SRC.mkdir(parents=True, exist_ok=True)

SETTINGS = {
    "seed": 42, "head_seed": 42,
    "gradient_checkpointing": True, "chunk_microbatch": 2,
    "backbone_dropout": 0.10, "head_dropout": 0.30,
    "decoder_upper_layers": 4,
    "adapter_bottleneck": 64, "adapter_dropout": 0.05,
    "adapter_train_layer_norm": True,
    "lorma_rank": 16, "lorma_alpha": 32, "lorma_dropout": 0.05,
    "projection_dim": 256, "context_hidden": 128, "attention_hidden": 64,
    "student_layers": [0, 7, 15],
    "smoke_steps": SMOKE_STEPS, "require_two_gpus": True,
    "run_resource_probes": RUN_RESOURCE_PROBES,
    "resource_budget_minutes": 12,
    "resource_candidates": [
        {"total_tokens": 256, "max_chunks": 4, "chunk_microbatch": 2},
        {"total_tokens": 512, "max_chunks": 4, "chunk_microbatch": 2},
        {"total_tokens": 512, "max_chunks": 8, "chunk_microbatch": 2},
        {"total_tokens": 512, "max_chunks": 8, "chunk_microbatch": 1},
    ],
}
CHUNK_POLICY_CANDIDATES = [
    {"total_tokens": 512, "overlap": 64, "max_chunks": 1, "policy": "head"},
    {"total_tokens": 512, "overlap": 64, "max_chunks": 1, "policy": "tail"},
    {"total_tokens": 256, "overlap": 32, "max_chunks": 4, "policy": "uniform"},
    {"total_tokens": 512, "overlap": 64, "max_chunks": 4, "policy": "uniform"},
    {"total_tokens": 512, "overlap": 64, "max_chunks": 4, "policy": "head_tail"},
    {"total_tokens": 512, "overlap": 64, "max_chunks": 8, "policy": "uniform"},
    {"total_tokens": 512, "overlap": 64, "max_chunks": 8, "policy": "head_tail"},
    {"total_tokens": 512, "overlap": 64, "max_chunks": 16, "policy": "uniform"},
]
CONFIG = {
    "run_id": RUN_ID, "working_root": str(ROOT),
    "output_root": "/kaggle/working", "input_root": "/kaggle/input",
    "prep_input": PREP_INPUT, "phase1_inputs": PHASE1_INPUTS,
    "cjpe_data_input": CJPE_DATA_INPUT, "hf_revision": HF_REVISION,
    "label_field": LABEL_FIELD, "run_model_probes": RUN_MODEL_PROBES,
    "settings": SETTINGS, "chunk_policy_candidates": CHUNK_POLICY_CANDIDATES,
}
print("Preparation:", RUN_ID)
print("Model probes:", RUN_MODEL_PROBES, "| Resource probes:", RUN_RESOURCE_PROBES)
print("Full training: disabled")

Preparation: phase4_cjpe_preparation_001
Model probes: True | Resource probes: True
Full training: disabled


## 2. Runtime

The setup cell uses Kaggle's PyTorch and installs any missing support
packages. The data loader reads the access token through the session secret.

In [2]:
import importlib.util
import subprocess

DEPENDENCY_ERROR = None
packages = {
    "numpy": "numpy", "pandas": "pandas", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "huggingface_hub": "huggingface_hub>=0.23",
}
missing = [package for module, package in packages.items() if importlib.util.find_spec(module) is None]
try:
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
    if importlib.util.find_spec("torch") is None:
        raise RuntimeError("PyTorch unavailable: select Kaggle T4 x2.")
except Exception as exc:
    DEPENDENCY_ERROR = str(exc)
    print("Dependency setup needs attention; the last cells can still export a failure ZIP.")
else:
    print("Supporting dependencies ready; installed PyTorch preserved.")

Supporting dependencies ready; installed PyTorch preserved.


## 3. Preparation and probe implementation

The cells below write the Phase 1 tokenizer and model definitions, CJPE
data loader, hierarchical classifier, probe utilities and course evaluator.
The loader records the official data and tokenizer hashes, audits repeated
IDs and creates a reusable per-case token cache.

In [3]:
%%writefile /kaggle/working/cs781_phase4_prep_src/phase1_utils.py
"""Small file and JSON helpers used by the Phase-1 notebooks."""
import hashlib
import json
import os
import pickle
import zipfile
from pathlib import Path


def sha256_file(path):
    digest = hashlib.sha256()
    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def text_hash(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def read_json(path):
    return json.loads(Path(path).read_text(encoding="utf-8"))


def atomic_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(temporary, path)


def atomic_pickle(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "wb") as file:
        pickle.dump(value, file, protocol=pickle.HIGHEST_PROTOCOL)
        file.flush()
        os.fsync(file.fileno())
    os.replace(temporary, path)


def checked_json(path, value):
    path = Path(path)
    if path.exists() and read_json(path) != value:
        raise ValueError(f"Saved settings differ: {path}. Use a new experiment ID.")
    atomic_json(path, value)


def write_source(path, content):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    raw = content.encode("utf-8") if isinstance(content, str) else content
    if path.exists():
        if path.read_bytes() != raw:
            raise ValueError(f"Saved source differs: {path}. Keep its code or use a new experiment ID.")
        return
    path.write_bytes(raw)


def make_bundle(folder, output_path, archive_root):
    folder, output_path, archive_root = Path(folder), Path(output_path), Path(archive_root)
    temporary = output_path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_STORED) as archive:
        for path in sorted(folder.rglob("*")):
            if path.is_file() and "__pycache__" not in path.parts and not path.name.endswith(".tmp"):
                archive.write(path, path.relative_to(archive_root))
    os.replace(temporary, output_path)
    return output_path

Overwriting /kaggle/working/cs781_phase4_prep_src/phase1_utils.py


In [4]:
%%writefile /kaggle/working/cs781_phase4_prep_src/a1_tokenizer.py
"""The assignment byte-BPE algorithm, with checkpoints between merges.

Pair counting, lexicographic tie breaking, non-overlapping replacements and
merge-order encoding are unchanged from the completed Phase-1/A3 tokenizer.
"""
import pickle
import re
import time
from collections import Counter
from pathlib import Path
from phase1_utils import atomic_pickle, sha256_file, text_hash


class A1BPETokenizer:
    def __init__(self):
        specials = ["<pad>", "<unk>", "<s>", "</s>"]
        self.vocab = [s.encode("utf-8") for s in specials]
        self.vocab += [bytes([b]) for b in range(256)]
        self.boundary_id = len(self.vocab)
        self.vocab.append(b"")
        self.names = [s.encode("utf-8") for s in specials]
        self.names += [bytes([b]) for b in range(256)]
        self.names.append(b"</w>")
        self.merges = {}
        self.vocab_size = len(self.vocab)

    def fit(self, text, vocab_size, checkpoint_path=None, checkpoint_merges=100,
            stop_after_merges=None):
        if vocab_size < 261:
            raise ValueError("Vocabulary must accommodate bytes and reserved tokens")
        signature = {"text_sha256": text_hash(text), "vocab_size": vocab_size,
                     "algorithm_source_sha256": sha256_file(__file__)}
        checkpoint_path = Path(checkpoint_path) if checkpoint_path is not None else None
        elapsed_before = 0.0
        if checkpoint_path is not None and checkpoint_path.exists():
            with open(checkpoint_path, "rb") as file:
                saved = pickle.load(file)
            if saved["signature"] != signature:
                raise ValueError("BPE checkpoint settings or source differ")
            self.__dict__.update(saved["tokenizer_state"])
            words = saved["words"]
            elapsed_before = saved["elapsed_seconds"]
            print("Resuming BPE vocabulary:", self.vocab_size, flush=True)
        else:
            if self.merges:
                raise ValueError("Use a fresh tokenizer or its saved fitting checkpoint")
            words = Counter(tuple(b + 4 for b in piece.encode("utf-8")) + (self.boundary_id,)
                            for piece in re.findall(r"\S+|\s+", text))
        started = time.monotonic()
        last_saved = started
        merges_this_call = 0
        commit_in_progress = False

        def snapshot():
            if checkpoint_path is not None:
                atomic_pickle(checkpoint_path, {"signature": signature,
                    "tokenizer_state": self.__dict__, "words": words,
                    "elapsed_seconds": elapsed_before + time.monotonic() - started})

        try:
            while len(self.vocab) < vocab_size:
                pair_counts = Counter()
                for symbols, frequency in words.items():
                    for pair in zip(symbols, symbols[1:]):
                        pair_counts[pair] += frequency
                if not pair_counts:
                    break
                pair = min(pair_counts, key=lambda p: (-pair_counts[p], self.names[p[0]], self.names[p[1]]))
                if pair_counts[pair] < 2:
                    break
                new_id = len(self.vocab)
                updated = Counter()
                for symbols, frequency in words.items():
                    result = []
                    j = 0
                    while j < len(symbols):
                        if j + 1 < len(symbols) and symbols[j:j + 2] == pair:
                            result.append(new_id)
                            j += 2
                        else:
                            result.append(symbols[j])
                            j += 1
                    updated[tuple(result)] += frequency
                # Commit the whole merge only after every word has been replaced.
                commit_in_progress = True
                self.vocab.append(self.vocab[pair[0]] + self.vocab[pair[1]])
                self.names.append(self.names[pair[0]] + self.names[pair[1]])
                self.merges[pair] = new_id
                self.vocab_size = len(self.vocab)
                words = updated
                commit_in_progress = False
                merges_this_call += 1
                if len(self.merges) % checkpoint_merges == 0 or time.monotonic() - last_saved >= 60:
                    snapshot()
                    print(f"BPE vocabulary {self.vocab_size:,}/{vocab_size:,} | "
                          f"elapsed {(elapsed_before + time.monotonic() - started)/60:.1f} min", flush=True)
                    last_saved = time.monotonic()
                if stop_after_merges is not None and merges_this_call >= stop_after_merges:
                    break
        finally:
            if not commit_in_progress:
                snapshot()
        return self

    def encode(self, text):
        result = []
        for piece in re.findall(r"\S+|\s+", text):
            symbols = [b + 4 for b in piece.encode("utf-8")] + [self.boundary_id]
            while len(symbols) > 1:
                candidates = [(self.merges[(a, b)], (a, b))
                              for a, b in zip(symbols, symbols[1:]) if (a, b) in self.merges]
                if not candidates:
                    break
                new_id, pair = min(candidates)
                merged = []
                j = 0
                while j < len(symbols):
                    if j + 1 < len(symbols) and (symbols[j], symbols[j + 1]) == pair:
                        merged.append(new_id)
                        j += 2
                    else:
                        merged.append(symbols[j])
                        j += 1
                symbols = merged
            result.extend(symbols)
        return result

    def decode(self, ids):
        return b"".join(self.vocab[i] for i in ids).decode("utf-8")

Overwriting /kaggle/working/cs781_phase4_prep_src/a1_tokenizer.py


In [5]:
%%writefile /kaggle/working/cs781_phase4_prep_src/phase1_decoder.py
"""Phase 1 decoder, adapted from completed Assignment 3 Q1."""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class LMConfig:
    vocab_size: int = 16000
    d_model: int = 1024
    n_layers: int = 16
    n_heads: int = 16
    d_ff: int = 4096
    max_seq_len: int = 2560
    dropout: float = 0.1
    norm_first: bool = True
    attention_impl: str = "sdpa"
    attention_scale: float = None
    embedding_scale: float = 32.0


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0,
                 attention_scale=None, attention_impl="sdpa"):
        super().__init__()
        assert d_model % n_heads == 0
        if attention_impl not in ("manual", "sdpa"):
            raise ValueError("attention_impl must be manual or sdpa")
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.attention_scale = 1.0 / math.sqrt(self.d_head) if attention_scale is None else float(attention_scale)
        self.attention_impl = attention_impl
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if attention_impl == "manual":
            self.register_buffer("causal_mask", torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)),
                                 persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        if self.attention_impl == "manual":
            scores = (q @ k.transpose(-2, -1)) * self.attention_scale
            scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
            weights = self.attn_dropout(torch.softmax(scores, dim=-1))
            out = weights @ v
        else:
            out = F.scaled_dot_product_attention(
                q, k, v, is_causal=True, scale=self.attention_scale,
                dropout_p=self.attn_dropout.p if self.training else 0.0,
            )
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    position = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * frequency)
    pe[:, 1::2] = torch.cos(position * frequency)
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(),
                                 nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.max_seq_len, cfg.dropout,
                                      cfg.attention_scale, cfg.attention_impl)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x):
        if self.norm_first:
            x = x + self.attn(self.ln1(x))
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model),
                             persistent=False)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

        # parameters() counts the tied matrix once.
        for parameter in self.parameters():
            if parameter.ndim > 1:
                nn.init.xavier_uniform_(parameter)
        for module in self.modules():
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, last_positions=None):
        B, T = idx.shape
        if T > self.cfg.max_seq_len:
            raise ValueError("Input exceeds the configured context length")
        x = self.tok_emb(idx) * self.cfg.embedding_scale + self.pos_emb[:T].unsqueeze(0)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        if last_positions is not None:
            x = x[torch.arange(B, device=idx.device), last_positions]
        return self.head(x)


class ModelWithLoss(nn.Module):
    """Compute loss on each GPU before DataParallel gathers the small outputs."""
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm = lm
        self.use_amp = use_amp

    def forward(self, x, y):
        # Enter autocast in each worker thread; do not move the model to half().
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits = self.lm(x)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), y.reshape(-1))
        return loss.unsqueeze(0)

Overwriting /kaggle/working/cs781_phase4_prep_src/phase1_decoder.py


In [6]:
%%writefile /kaggle/working/cs781_phase4_prep_src/phase1_shared.py
"""Three from-scratch Phase 1 Transformers; course blocks with cited extensions.

SDPA Boolean masks use True for ALLOWED connections. All blocks are pre-norm.
Training uses full teacher-forced sequences. KV caches are inference-only.
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    architecture: str = "encoder_decoder"
    vocab_size: int = 16000
    d_model: int = 1024
    n_heads: int = 16
    n_layers: int = 16
    encoder_layers: int = 8
    decoder_layers: int = 6
    n_kv_heads: int = 4
    d_ff: int = 4096
    max_seq_len: int = 2560
    dropout: float = 0.1
    window_size: int = 512
    query_chunk_size: int = 128


def configuration(architecture):
    if architecture not in ("encoder_decoder", "prefix", "efficient"):
        raise ValueError("Unknown architecture")
    return ModelConfig(architecture=architecture, d_ff=4864 if architecture == "efficient" else 4096)


def positions(max_len, width):
    p = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, width, 2).float() * (-math.log(10000.0) / width))
    result = torch.zeros(max_len, width)
    result[:, 0::2] = torch.sin(p * frequency)
    result[:, 1::2] = torch.cos(p * frequency)
    return result


def valid_keys(lengths, width):
    return torch.arange(width, device=lengths.device)[None, :] < lengths[:, None]


class Attention(nn.Module):
    def __init__(self, cfg, kv_heads=None):
        super().__init__()
        self.heads = cfg.n_heads
        self.kv_heads = cfg.n_heads if kv_heads is None else kv_heads
        assert cfg.d_model % self.heads == 0 and self.heads % self.kv_heads == 0
        self.head_dim = cfg.d_model // self.heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def query(self, x):
        b, t, _ = x.shape
        return self.q_proj(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)

    def keys_values(self, x):
        b, t, _ = x.shape
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        return k, v

    def attend(self, q, k, v, allowed=None, causal=False):
        # A3 Q2: repeat each KV head for its group of query heads.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k, v = k.repeat_interleave(repeats, dim=1), v.repeat_interleave(repeats, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0, scale=1 / math.sqrt(self.head_dim))
        b, _, t, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(b, t, -1)
        return self.resid_dropout(self.out_proj(out))

    def forward(self, x, memory=None, allowed=None, causal=False):
        k, v = self.keys_values(x if memory is None else memory)
        return self.attend(self.query(x), k, v, allowed, causal)

    def local(self, x, window, chunk, lengths=None):
        # Query tiles see only nearby keys. No full T-by-T score matrix.
        q = self.query(x)
        k, v = self.keys_values(x)
        t = x.shape[1]
        pieces = []
        for start in range(0, t, chunk):
            stop = min(t, start + chunk)
            first = max(0, start - window + 1)
            qi = torch.arange(start, stop, device=x.device)
            ki = torch.arange(first, stop, device=x.device)
            allowed = (ki[None, :] <= qi[:, None]) & (ki[None, :] > qi[:, None] - window)
            allowed = allowed[None, None, :, :]
            if lengths is not None:
                allowed = allowed & (ki[None, :] < lengths[:, None])[:, None, None, :]
            pieces.append(self.attend(q[:, :, start:stop], k[:, :, first:stop], v[:, :, first:stop], allowed))
        return torch.cat(pieces, dim=1)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
                                 nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout))

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg), FeedForward(cfg)

    def forward(self, x, allowed):
        x = x + self.attn(self.ln1(x), allowed=allowed)
        return x + self.ffn(self.ln2(x))


class DecoderBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2, self.ln3 = (nn.LayerNorm(cfg.d_model) for _ in range(3))
        self.self_attn, self.cross_attn, self.ffn = Attention(cfg), Attention(cfg), FeedForward(cfg)

    def forward(self, x, memory, allowed):
        x = x + self.self_attn(self.ln1(x), causal=True)
        x = x + self.cross_attn(self.ln2(x), memory=memory, allowed=allowed)
        return x + self.ffn(self.ln3(x))


class PrefixBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg), FeedForward(cfg)

    def forward(self, prompt, decoder, prefix_allowed):
        # The prompt cannot see decoder tokens; decoder queries see both parts.
        p = self.ln1(prompt)
        d = self.ln1(decoder)
        pk, pv = self.attn.keys_values(p)
        dk, dv = self.attn.keys_values(d)
        prompt = prompt + self.attn.attend(self.attn.query(p), pk, pv, prefix_allowed)
        t = decoder.shape[1]
        triangular = torch.ones(t, t, dtype=torch.bool, device=decoder.device).tril()
        allowed = torch.cat([prefix_allowed.expand(-1, -1, t, -1),
                             triangular[None, None].expand(decoder.shape[0], 1, -1, -1)], dim=-1)
        decoder = decoder + self.attn.attend(self.attn.query(d), torch.cat([pk, dk], dim=2),
                                            torch.cat([pv, dv], dim=2), allowed)
        return prompt + self.ffn(self.ln2(prompt)), decoder + self.ffn(self.ln2(decoder))


class EfficientBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg, cfg.n_kv_heads), FeedForward(cfg)
        self.window, self.chunk = cfg.window_size, cfg.query_chunk_size

    def forward(self, x, lengths=None):
        x = x + self.attn.local(self.ln1(x), self.window, self.chunk, lengths)
        return x + self.ffn(self.ln2(x))


class BaseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", positions(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def initialize(self):
        for p in self.parameters():
            if p.ndim > 1:
                nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def embed(self, ids, offset=None, role=None):
        t = ids.shape[1]
        p = torch.arange(t, device=ids.device)[None, :]
        if offset is not None:
            p = p + offset[:, None]
        x = self.tok_emb(ids) * math.sqrt(self.cfg.d_model) + self.pos_emb[p]
        if role is not None:
            x = x + self.segment_emb.weight[role]
        return x


class EncoderDecoderLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.encoder = nn.ModuleList([EncoderBlock(cfg) for _ in range(cfg.encoder_layers)])
        self.decoder = nn.ModuleList([DecoderBlock(cfg) for _ in range(cfg.decoder_layers)])
        self.encoder_norm, self.decoder_norm = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.initialize()

    def encode(self, prompt, lengths):
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        x = self.embed(prompt)
        for block in self.encoder:
            x = block(x, allowed)
        return self.encoder_norm(x), allowed

    def forward(self, prompt, decoder_ids, lengths=None):
        if prompt.shape[1] + decoder_ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Prompt plus decoder exceeds context budget")
        if lengths is None:
            lengths = torch.full((prompt.shape[0],), prompt.shape[1], device=prompt.device, dtype=torch.long)
        memory, allowed = self.encode(prompt, lengths)
        x = self.embed(decoder_ids)
        for block in self.decoder:
            x = block(x, memory, allowed)
        return self.head(self.decoder_norm(x))

    def start_generation(self, prompt, lengths):
        memory, allowed = self.encode(prompt, lengths)
        layers = []
        for block in self.decoder:
            k, v = block.cross_attn.keys_values(memory)
            layers.append({"cross_k": k, "cross_v": v,
                           "k": k[:, :, :0], "v": v[:, :, :0]})
        cache = {"layers": layers, "allowed": allowed, "lengths": lengths,
                 "decoder_position": 0, "next_positions": lengths.clone()}
        bos = torch.full((prompt.shape[0], 1), 2, dtype=torch.long, device=prompt.device)
        return self.generation_step(bos, cache)

    def generation_step(self, ids, cache):
        position = cache["decoder_position"]
        x = self.embed(ids, offset=torch.full_like(cache["lengths"], position))
        for block, layer in zip(self.decoder, cache["layers"]):
            h = block.ln1(x)
            k, v = block.self_attn.keys_values(h)
            layer["k"], layer["v"] = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            # One new query may see all cached decoder keys, including itself.
            x = x + block.self_attn.attend(block.self_attn.query(h), layer["k"], layer["v"])
            x = x + block.cross_attn.attend(block.cross_attn.query(block.ln2(x)),
                                          layer["cross_k"], layer["cross_v"], cache["allowed"])
            x = x + block.ffn(block.ln3(x))
        cache["decoder_position"] += 1
        cache["next_positions"] = cache["lengths"] + cache["decoder_position"]
        return self.head(self.decoder_norm(x))[:, -1], cache


class PrefixLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.segment_emb = nn.Embedding(2, cfg.d_model)
        self.blocks = nn.ModuleList([PrefixBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, prompt, decoder_ids, lengths=None):
        if prompt.shape[1] + decoder_ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Prompt plus decoder exceeds context budget")
        if lengths is None:
            lengths = torch.full((prompt.shape[0],), prompt.shape[1], device=prompt.device, dtype=torch.long)
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        p, d = self.embed(prompt, role=0), self.embed(decoder_ids, offset=lengths, role=1)
        for block in self.blocks:
            p, d = block(p, d, allowed)
        return self.head(self.final_norm(d))

    def start_generation(self, prompt, lengths):
        allowed = valid_keys(lengths, prompt.shape[1])[:, None, None, :]
        x = self.embed(prompt, role=0)
        layers = []
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            layers.append({"prefix_k": k, "prefix_v": v, "k": k[:, :, :0], "v": v[:, :, :0]})
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
        cache = {"layers": layers, "allowed": allowed, "lengths": lengths,
                 "decoder_position": 0, "next_positions": lengths.clone()}
        bos = torch.full((prompt.shape[0], 1), 2, dtype=torch.long, device=prompt.device)
        return self.generation_step(bos, cache)

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"], role=1)
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            layer["k"], layer["v"] = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            allowed = torch.cat([cache["allowed"],
                torch.ones((ids.shape[0], 1, 1, layer["k"].shape[2]), device=ids.device, dtype=torch.bool)], dim=-1)
            x = x + block.attn.attend(block.attn.query(h),
                torch.cat([layer["prefix_k"], layer["k"]], dim=2),
                torch.cat([layer["prefix_v"], layer["v"]], dim=2), allowed)
            x = x + block.ffn(block.ln2(x))
        cache["decoder_position"] += 1
        cache["next_positions"] = cache["lengths"] + cache["decoder_position"]
        return self.head(self.final_norm(x))[:, -1], cache


class EfficientLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.blocks = nn.ModuleList([EfficientBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, ids, lengths=None):
        if ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Input exceeds context budget")
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, lengths)
        return self.head(self.final_norm(x))

    def start_generation(self, prompt, lengths):
        x = self.embed(prompt)
        layers = []
        keep = self.cfg.window_size - 1
        indices = lengths[:, None] - keep + torch.arange(keep, device=prompt.device)[None, :]
        gather = indices.clamp_min(0)[:, None, :, None]
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            shape = (prompt.shape[0], k.shape[1], keep, k.shape[-1])
            layers.append({"k": torch.gather(k, 2, gather.expand(shape)),
                           "v": torch.gather(v, 2, gather.expand(shape))})
            x = x + block.attn.local(h, block.window, block.chunk, lengths)
            x = x + block.ffn(block.ln2(x))
        last = self.final_norm(x)[torch.arange(x.shape[0], device=x.device), lengths - 1]
        cache = {"layers": layers, "lengths": lengths, "next_positions": lengths.clone(),
                 "key_positions": indices}
        return self.head(last), cache

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"])
        key_positions = torch.cat([cache["key_positions"], cache["next_positions"][:, None]], dim=1)
        allowed = (key_positions >= 0)[:, None, None, :]
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            k, v = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
            layer["k"], layer["v"] = k[:, :, 1:], v[:, :, 1:]
        cache["key_positions"] = key_positions[:, 1:]
        cache["next_positions"] = cache["next_positions"] + 1
        return self.head(self.final_norm(x))[:, -1], cache


def build_model(cfg):
    constructors = {"encoder_decoder": EncoderDecoderLM, "prefix": PrefixLM, "efficient": EfficientLM}
    return constructors[cfg.architecture](cfg)


def select_cache(cache, indices):
    """Remove finished rows only when the active batch shrinks."""
    if isinstance(cache, torch.Tensor):
        return cache.index_select(0, indices)
    if isinstance(cache, dict):
        return {k: select_cache(v, indices) for k, v in cache.items()}
    if isinstance(cache, list):
        return [select_cache(v, indices) for v in cache]
    return cache


def logits_and_labels(lm, x, y, prefix_length):
    if lm.cfg.architecture == "efficient":
        return lm(x), y
    p = int(prefix_length)
    if not 0 < p < x.shape[1]:
        raise ValueError("Need a nonempty prefix and continuation")
    labels = x[:, p:]
    bos = torch.full_like(labels[:, :1], 2)
    decoder = torch.cat([bos, labels[:, :-1]], dim=1)
    return lm(x[:, :p], decoder), labels


class ModelWithLoss(nn.Module):
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm, self.use_amp = lm, use_amp

    def forward(self, x, y, prefixes):
        # DataParallel scatters this per-example tensor along with x and y.
        prefix_length = int(prefixes[0].item())
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits, labels = logits_and_labels(self.lm, x, y, prefix_length)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), labels.reshape(-1))
        return loss.unsqueeze(0)

Overwriting /kaggle/working/cs781_phase4_prep_src/phase1_shared.py


In [7]:
%%writefile /kaggle/working/cs781_phase4_prep_src/phase1_efficient.py
"""Efficient causal LM: assignment GQA with actual sliding-window attention.

Course Module 2.3 pages 155-156, 167-170 and 180-185.
A3 Q2 supplies the KV-head sharing algorithm. SDPA Boolean True means allowed.
Training uses next-token targets. KV caches are inference-only.
"""
import math
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F


@dataclass
class ModelConfig:
    architecture: str = "efficient"
    vocab_size: int = 16000
    d_model: int = 1024
    n_heads: int = 16
    n_layers: int = 16
    n_kv_heads: int = 4
    d_ff: int = 4864
    max_seq_len: int = 2560
    dropout: float = 0.1
    window_size: int = 512
    query_chunk_size: int = 128

    def __post_init__(self):
        if self.architecture != "efficient":
            raise ValueError("This notebook implements the efficient architecture only")
        assert self.d_model % self.n_heads == 0 and self.n_heads % self.n_kv_heads == 0
        assert min(self.vocab_size - 4, self.n_layers, self.d_ff, self.max_seq_len,
                   self.window_size, self.query_chunk_size) > 0
        assert 0 <= self.dropout < 1


def configuration(architecture="efficient"):
    return ModelConfig(architecture=architecture)



def positions(max_len, width):
    p = torch.arange(max_len).unsqueeze(1).float()
    frequency = torch.exp(torch.arange(0, width, 2).float() * (-math.log(10000.0) / width))
    result = torch.zeros(max_len, width)
    result[:, 0::2] = torch.sin(p * frequency)
    result[:, 1::2] = torch.cos(p * frequency)
    return result


class Attention(nn.Module):
    def __init__(self, cfg, kv_heads=None):
        super().__init__()
        self.heads = cfg.n_heads
        self.kv_heads = cfg.n_heads if kv_heads is None else kv_heads
        assert cfg.d_model % self.heads == 0 and self.heads % self.kv_heads == 0
        self.head_dim = cfg.d_model // self.heads
        self.q_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.k_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(cfg.d_model, self.kv_heads * self.head_dim, bias=False)
        self.out_proj = nn.Linear(cfg.d_model, cfg.d_model, bias=False)
        self.dropout = cfg.dropout
        self.resid_dropout = nn.Dropout(cfg.dropout)

    def query(self, x):
        b, t, _ = x.shape
        return self.q_proj(x).view(b, t, self.heads, self.head_dim).transpose(1, 2)

    def keys_values(self, x):
        b, t, _ = x.shape
        k = self.k_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(x).view(b, t, self.kv_heads, self.head_dim).transpose(1, 2)
        return k, v

    def attend(self, q, k, v, allowed=None, causal=False):
        # A3 Q2: repeat each KV head for its group of query heads.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k, v = k.repeat_interleave(repeats, dim=1), v.repeat_interleave(repeats, dim=1)
        out = F.scaled_dot_product_attention(q, k, v, attn_mask=allowed, is_causal=causal,
            dropout_p=self.dropout if self.training else 0.0, scale=1 / math.sqrt(self.head_dim))
        b, _, t, _ = out.shape
        out = out.transpose(1, 2).contiguous().view(b, t, -1)
        return self.resid_dropout(self.out_proj(out))

    def forward(self, x, memory=None, allowed=None, causal=False):
        k, v = self.keys_values(x if memory is None else memory)
        return self.attend(self.query(x), k, v, allowed, causal)

    def local(self, x, window, chunk, lengths=None):
        # Query tiles see only nearby keys. No full T-by-T score matrix.
        q = self.query(x)
        k, v = self.keys_values(x)
        t = x.shape[1]
        pieces = []
        for start in range(0, t, chunk):
            stop = min(t, start + chunk)
            first = max(0, start - window + 1)
            qi = torch.arange(start, stop, device=x.device)
            ki = torch.arange(first, stop, device=x.device)
            allowed = (ki[None, :] <= qi[:, None]) & (ki[None, :] > qi[:, None] - window)
            allowed = allowed[None, None, :, :]
            if lengths is not None:
                allowed = allowed & (ki[None, :] < lengths[:, None])[:, None, None, :]
            pieces.append(self.attend(q[:, :, start:stop], k[:, :, first:stop], v[:, :, first:stop], allowed))
        return torch.cat(pieces, dim=1)


class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(cfg.d_model, cfg.d_ff), nn.GELU(),
                                 nn.Linear(cfg.d_ff, cfg.d_model), nn.Dropout(cfg.dropout))

    def forward(self, x):
        return self.net(x)


class EfficientBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(cfg.d_model), nn.LayerNorm(cfg.d_model)
        self.attn, self.ffn = Attention(cfg, cfg.n_kv_heads), FeedForward(cfg)
        self.window, self.chunk = cfg.window_size, cfg.query_chunk_size

    def forward(self, x, lengths=None):
        x = x + self.attn.local(self.ln1(x), self.window, self.chunk, lengths)
        return x + self.ffn(self.ln2(x))


class BaseModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.register_buffer("pos_emb", positions(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight

    def initialize(self):
        for p in self.parameters():
            if p.ndim > 1:
                nn.init.xavier_uniform_(p)
        for m in self.modules():
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def embed(self, ids, offset=None):
        t = ids.shape[1]
        p = torch.arange(t, device=ids.device)[None, :]
        if offset is not None:
            p = p + offset[:, None]
        x = self.tok_emb(ids) * math.sqrt(self.cfg.d_model) + self.pos_emb[p]
        return x


class EfficientLM(BaseModel):
    def __init__(self, cfg):
        super().__init__(cfg)
        self.blocks = nn.ModuleList([EfficientBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.initialize()

    def forward(self, ids, lengths=None):
        if ids.shape[1] > self.cfg.max_seq_len:
            raise ValueError("Input exceeds context budget")
        x = self.embed(ids)
        for block in self.blocks:
            x = block(x, lengths)
        return self.head(self.final_norm(x))

    def start_generation(self, prompt, lengths):
        x = self.embed(prompt)
        layers = []
        keep = self.cfg.window_size - 1
        indices = lengths[:, None] - keep + torch.arange(keep, device=prompt.device)[None, :]
        gather = indices.clamp_min(0)[:, None, :, None]
        for block in self.blocks:
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            shape = (prompt.shape[0], k.shape[1], keep, k.shape[-1])
            layers.append({"k": torch.gather(k, 2, gather.expand(shape)),
                           "v": torch.gather(v, 2, gather.expand(shape))})
            x = x + block.attn.local(h, block.window, block.chunk, lengths)
            x = x + block.ffn(block.ln2(x))
        last = self.final_norm(x)[torch.arange(x.shape[0], device=x.device), lengths - 1]
        cache = {"layers": layers, "lengths": lengths, "next_positions": lengths.clone(),
                 "key_positions": indices}
        return self.head(last), cache

    def generation_step(self, ids, cache):
        x = self.embed(ids, offset=cache["next_positions"])
        key_positions = torch.cat([cache["key_positions"], cache["next_positions"][:, None]], dim=1)
        allowed = (key_positions >= 0)[:, None, None, :]
        for block, layer in zip(self.blocks, cache["layers"]):
            h = block.ln1(x)
            k, v = block.attn.keys_values(h)
            k, v = torch.cat([layer["k"], k], dim=2), torch.cat([layer["v"], v], dim=2)
            x = x + block.attn.attend(block.attn.query(h), k, v, allowed)
            x = x + block.ffn(block.ln2(x))
            layer["k"], layer["v"] = k[:, :, 1:], v[:, :, 1:]
        cache["key_positions"] = key_positions[:, 1:]
        cache["next_positions"] = cache["next_positions"] + 1
        return self.head(self.final_norm(x))[:, -1], cache


def select_cache(cache, indices):
    """Remove finished rows only when the active batch shrinks."""
    if isinstance(cache, torch.Tensor):
        return cache.index_select(0, indices)
    if isinstance(cache, dict):
        return {k: select_cache(v, indices) for k, v in cache.items()}
    if isinstance(cache, list):
        return [select_cache(v, indices) for v in cache]
    return cache


class ModelWithLoss(nn.Module):
    def __init__(self, lm, use_amp=True):
        super().__init__()
        self.lm, self.use_amp = lm, use_amp

    def forward(self, x, y, prefixes):
        # DataParallel scatters this per-example tensor along with x and y.
        with torch.autocast("cuda", dtype=torch.float16, enabled=self.use_amp and x.is_cuda):
            logits, labels = logits_and_labels(self.lm, x, y, 0)
            loss = F.cross_entropy(logits.reshape(-1, self.lm.cfg.vocab_size), labels.reshape(-1))
        return loss.unsqueeze(0)


def build_model(cfg):
    return EfficientLM(cfg)


def logits_and_labels(lm, x, y, prefix_length=0):
    # y is the token immediately after each input position in x.
    return lm(x), y

Overwriting /kaggle/working/cs781_phase4_prep_src/phase1_efficient.py


In [8]:
%%writefile /kaggle/working/cs781_phase4_prep_src/cjpe_data.py
"""CJPE input discovery, official data, saved BPE, document caches and audits."""
import collections
import hashlib
import json
import math
import os
import pickle
import re
import shutil
import time
import zipfile
from pathlib import Path
import numpy as np


def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(8 * 2**20), b""):
            h.update(block)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_name(path.name + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    temporary.replace(path)


def read_json(path):
    def pairs(items):
        result = {}
        for k, v in items:
            if k in result:
                raise ValueError("Duplicate JSON key: " + str(k))
            result[k] = v
        return result
    def invalid(x):
        raise ValueError("Nonfinite JSON value: " + x)
    return json.loads(Path(path).read_text(encoding="utf-8-sig"),
                      object_pairs_hook=pairs, parse_constant=invalid)


def safe_error(exc):
    return re.sub(r"hf_[A-Za-z0-9]+", "[redacted]", str(exc))[:3000]


def discover(filename, input_root, supplied=None, hint=None, extraction_root=None):
    """Find one unambiguous file; extract only requested members and companions."""
    base = Path(supplied) if supplied else Path(input_root)
    hints = [hint] if isinstance(hint, str) else list(hint or [])
    if supplied and not base.exists():
        raise FileNotFoundError("Input path does not exist: " + str(base))
    candidates = [base] if base.is_file() and base.name == filename else (
        list(base.rglob(filename)) if base.is_dir() else [])
    if hints and not supplied:
        candidates = [p for p in candidates if any(h.lower() in str(p).lower() for h in hints)]
    candidates = sorted(set(p.resolve() for p in candidates))
    if len(candidates) == 1:
        return candidates[0]
    if len(candidates) > 1:
        raise ValueError("Several " + filename + " files found. Set its exact input path: " +
                         " | ".join(str(p) for p in candidates[:10]))
    archives = [base] if base.is_file() and base.suffix.lower() == ".zip" else (
        sorted(base.rglob("*.zip")) if base.is_dir() else [])
    matches = []
    for archive in archives:
        try:
            with zipfile.ZipFile(archive) as z:
                for info in z.infolist():
                    if not info.is_dir() and Path(info.filename).name == filename:
                        if not hints or supplied or any(h.lower() in (str(archive)+"/"+info.filename).lower() for h in hints):
                            matches.append((archive, info.filename))
        except zipfile.BadZipFile:
            continue
    if len(matches) != 1:
        raise FileNotFoundError("Need exactly one " + filename + " for " + str(hint or supplied) +
                                "; archive matches=" + str(len(matches)) +
                                ". Attach the original dataset or set its exact path.")
    archive, member = matches[0]
    target_root = Path(extraction_root) / hashlib.sha256(str(archive.resolve()).encode()).hexdigest()[:12]
    parent = Path(member).parent
    with zipfile.ZipFile(archive) as z:
        wanted = [member]
        for companion in ("cache_manifest.json", "sources/phase1_decoder.py", "sources/architecture_models.py"):
            name = str(parent / companion)
            if name in z.namelist():
                wanted.append(name)
        for name in wanted:
            target = (target_root / name).resolve()
            if not target.is_relative_to(target_root.resolve()):
                raise ValueError("Unsafe archive member.")
            info = z.getinfo(name)
            if shutil.disk_usage(target_root.parent if target_root.parent.exists() else Path.cwd()).free < info.file_size + 2**28:
                raise RuntimeError("Insufficient disk for the selected checkpoint.")
            target.parent.mkdir(parents=True, exist_ok=True)
            with z.open(name) as src, open(str(target) + ".tmp", "wb") as dst:
                shutil.copyfileobj(src, dst, 8 * 2**20)
            Path(str(target) + ".tmp").replace(target)
    return target_root / member


def official_data(root, requested_revision=None, local_input=None):
    """Pin one actual repository commit before downloading only Phase 4 JSON."""
    root = Path(root)
    root.mkdir(parents=True, exist_ok=True)
    manifest_path = root / "download_manifest.json"
    if manifest_path.exists():
        old = read_json(manifest_path)
        if requested_revision not in (None, "main", old.get("revision")):
            raise ValueError("This working data cache belongs to a different requested revision.")
        if all((root / key).is_file() and sha256(root / key) == value for key, value in old["sha256"].items()):
            print("Reused verified official CJPE files at commit", old["revision"], flush=True)
            return root, old
        raise ValueError("Existing CJPE files differ from their manifest. Use a fresh RUN_ID.")
    relative = {"train": "Train/train.json", "dev": "Train/dev.json", "test": "Test/test.json"}
    if local_input:
        p = Path(local_input)
        if p.is_file() and p.suffix.lower() == ".zip":
            destination = root / "_attached"
            with zipfile.ZipFile(p) as z:
                for split, suffix in relative.items():
                    choices = [n for n in z.namelist() if n.endswith(suffix) and "Phase_4_CJPE" in n]
                    if len(choices) != 1:
                        raise ValueError("Attached CJPE archive needs one Phase_4_CJPE/" + suffix)
                    target = root / suffix
                    target.parent.mkdir(parents=True, exist_ok=True)
                    with z.open(choices[0]) as src, open(target, "wb") as dst:
                        shutil.copyfileobj(src, dst)
        else:
            for suffix in relative.values():
                choices = [q for q in p.rglob(Path(suffix).name) if str(q).endswith(suffix)]
                if len(choices) != 1:
                    raise ValueError("Choose a directory containing exactly one " + suffix)
                target = root / suffix
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(choices[0], target)
        identity = {"repo_id": "Exploration-Lab/CS-781-Capstone", "revision": "attached_local_copy",
                    "provenance": "user-supplied; byte hashes recorded, remote revision unverified"}
    else:
        from huggingface_hub import HfApi, hf_hub_download
        token = os.environ.get("HF_TOKEN")
        if not token:
            try:
                from kaggle_secrets import UserSecretsClient
                token = UserSecretsClient().get_secret("HF_TOKEN")
            except Exception:
                raise RuntimeError("Enable Internet and grant this notebook access to Kaggle secret HF_TOKEN.") from None
        repo = "Exploration-Lab/CS-781-Capstone"
        try:
            api = HfApi(token=token)
            revision = api.dataset_info(repo, revision=requested_revision or "main").sha
            files = api.list_repo_files(repo_id=repo, repo_type="dataset", revision=revision)
            resolved = {}
            for split, suffix in relative.items():
                exact = "Phase_4_CJPE/" + suffix
                choices = [n for n in files if n == exact]
                if not choices:
                    choices = [n for n in files if n.lower() == exact.lower() or n.lower().endswith("/" + exact.lower())]
                if not choices:
                    choices = [n for n in files if "phase4cjpe" in re.sub(r"[^a-z0-9]", "", n.lower())
                               and n.lower().endswith("/" + suffix.lower())]
                if len(choices) != 1:
                    preview = [n for n in files if "cjpe" in n.lower() or "phase_4" in n.lower()]
                    write_json(root / "phase4_repository_paths.json", preview)
                    raise ValueError("Could not resolve " + exact + "; repository paths saved for inspection.")
                downloaded = hf_hub_download(repo_id=repo, repo_type="dataset", filename=choices[0],
                    revision=revision, token=token, local_dir=str(root / "_download"))
                resolved[split] = choices[0]
                target = root / suffix
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(downloaded, target)
            identity = {"repo_id": repo, "revision": revision, "resolved_files": resolved,
                        "provenance": "official pinned download"}
        except Exception as exc:
            raise RuntimeError("Official CJPE download failed: " + safe_error(exc)) from None
        finally:
            token = None
    identity["sha256"] = {suffix: sha256(root / suffix) for suffix in relative.values()}
    write_json(manifest_path, identity)
    return root, identity


def normalize_records(raw, split, label_field=None):
    """Accept explicit list/dictionary schemas without guessing ACCEPT/REJECT mapping."""
    if isinstance(raw, dict):
        wrappers = [k for k in ("data", "records", "examples", "cases")
                    if isinstance(raw.get(k), list) and "text" not in raw]
        if len(wrappers) == 1:
            raw = raw[wrappers[0]]
    if isinstance(raw, list):
        items = [(None, r) for r in raw]
    elif isinstance(raw, dict) and "id" in raw and "text" in raw:
        items = [(None, raw)]
    elif isinstance(raw, dict):
        items = list(raw.items())
    else:
        raise ValueError(split + ": expected a list or an ID-to-record mapping.")
    documents, seen, schema = [], set(), collections.Counter()
    for row_index, (key, value) in enumerate(items):
        if split == "test" and key is not None and isinstance(value, (str, list)):
            value = {"id": key, "text": value}
        if not isinstance(value, dict):
            raise ValueError(split + ": each case must contain text and an ID.")
        case_id = value.get("id", key)
        if not isinstance(case_id, str) or not case_id.strip():
            raise ValueError(split + ": case IDs must be nonempty strings; no ID conversion is applied.")
        if key is not None and key != case_id:
            raise ValueError(split + ": dictionary key and record ID disagree.")
        if case_id in seen:
            schema["rows_repeating_an_original_case_id"] += 1
        seen.add(case_id)
        text = value.get("text")
        if isinstance(text, list) and all(isinstance(x, str) for x in text):
            text = "\n".join(text)
            schema["text_list_joined_with_newlines"] += 1
        elif isinstance(text, str):
            schema["text_string"] += 1
        else:
            raise ValueError(split + "/" + case_id + ": text must be a string or ordered string list.")
        if not text.strip():
            raise ValueError(split + "/" + case_id + ": empty document.")
        gold = None
        if split != "test":
            fields = [label_field] if label_field else [
                k for k in ("gold", "label", "labels", "target", "outcome") if k in value]
            if not fields or any(k not in value for k in fields):
                raise ValueError(split + ": binary label field not found; keys=" + str(sorted(value)))
            labels = []
            for field in fields:
                v = value[field]
                if isinstance(v, list) and len(v) == 1:
                    v = v[0]
                    schema["singleton_label_list_unwrapped"] += 1
                if isinstance(v, str) and v in ("0", "1"):
                    v = int(v)
                    schema["numeric_string_label_converted"] += 1
                if isinstance(v, bool) or not isinstance(v, int) or v not in (0, 1):
                    raise ValueError(split + ": need a scalar binary 0/1 target; no word-label mapping is assumed.")
                labels.append(v)
            if len(set(labels)) != 1:
                raise ValueError(split + ": candidate label fields disagree.")
            gold = labels[0]
            schema["label_field:" + fields[0]] += 1
        schema["records"] += 1
        # Official train/dev may repeat a case ID. Preserve every provided row and raw ID;
        # use a separate identity for caches, local evaluation and teacher-target alignment.
        row_uid = split + ":" + str(row_index).zfill(8) + ":" + case_id
        documents.append({"id": case_id, "row_uid": row_uid, "row_index": row_index,
                          "split": split, "text": text, "gold": gold})
    if not documents:
        raise ValueError(split + ": no cases.")
    return documents, dict(schema)


def load_documents(root, label_field=None):
    documents, schemas = [], {}
    for split, suffix in (("train", "Train/train.json"), ("dev", "Train/dev.json"), ("test", "Test/test.json")):
        part, schemas[split] = normalize_records(read_json(Path(root) / suffix), split, label_field)
        documents.extend(part)
    return documents, schemas


def evaluation_id(document):
    """Local held-out rows are unique; submission IDs stay exactly as supplied."""
    return document["id"] if document["split"] == "test" else document["row_uid"]


def audit_record_ids(documents, output):
    """Audit repeats before BPE; no row removal, label merging, or raw-ID rewriting."""
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    report = {"record_identity_version": 2,
        "policy": "Preserve all supplied rows and original IDs; use split/row-index/raw-ID row_uid internally.",
        "local_evaluation_unit": "provided development rows",
        "test_output_identity": "unchanged official case ID",
        "submission_blockers": [], "review_warnings": [], "splits": {}}
    examples = []
    for split in ("train", "dev", "test"):
        groups = collections.defaultdict(list)
        for d in documents:
            if d["split"] == split:
                groups[d["id"]].append(d)
        collisions = []
        for case_id, rows in sorted(groups.items()):
            if len(rows) < 2:
                continue
            exact, normalized = set(), collections.defaultdict(list)
            members = []
            for d in rows:
                exact_sha = hashlib.sha256(d["text"].encode()).hexdigest()
                norm_sha = hashlib.sha256(re.sub(r"\s+", " ", d["text"]).strip().casefold().encode()).hexdigest()
                exact.add(exact_sha)
                normalized[norm_sha].append(d)
                members.append({"row_uid": d["row_uid"], "row_index": d["row_index"],
                    "gold": d["gold"], "characters": len(d["text"]), "text_sha256": exact_sha})
            conflicts = [{"normalized_text_sha256": key,
                "row_uids": [d["row_uid"] for d in same],
                "gold_labels": sorted({d["gold"] for d in same if d["gold"] is not None})}
                for key, same in normalized.items()
                if len({d["gold"] for d in same if d["gold"] is not None}) > 1]
            collisions.append({"original_case_id": case_id, "rows": len(rows),
                "distinct_exact_texts": len(exact), "distinct_normalized_texts": len(normalized),
                "label_counts": dict(collections.Counter(str(d["gold"]) for d in rows)),
                "same_normalized_text_conflicting_labels": conflicts, "members": members})
            if len(examples) < 20:
                examples.append({"split": split, "original_case_id": case_id,
                    "members": [{"row_uid": d["row_uid"], "gold": d["gold"],
                        "beginning": d["text"][:200], "end": d["text"][-200:]} for d in rows[:4]]})
        report["splits"][split] = {"provided_rows": sum(map(len, groups.values())),
            "unique_original_case_ids": len(groups), "repeated_original_id_groups": len(collisions),
            "extra_rows_with_repeated_ids": sum(g["rows"]-1 for g in collisions),
            "collision_groups": collisions}
        if split == "test" and collisions:
            report["submission_blockers"].append(
                "Official test IDs repeat; one prediction per official ID is ambiguous. Inspect record_id_audit.json.")
        elif collisions:
            report["review_warnings"].append(split + " contains " + str(len(collisions)) +
                " repeated-ID groups; all rows retained and reported separately.")
    write_json(output/"record_id_audit.json", report)
    write_json(output/"repeated_id_examples.json", examples)
    return report


def load_tokenizer(path, expected_hash=None):
    from a1_tokenizer import A1BPETokenizer
    path = Path(path)
    manifest = read_json(path.parent / "cache_manifest.json")
    digest = sha256(path)
    if digest != manifest["tokenizer_state_sha256"] or (expected_hash and digest != expected_hash):
        raise ValueError("Use the original prep-2 tokenizer and its matching cache_manifest.json.")
    with open(path, "rb") as f:
        saved = pickle.load(f)  # Only the user's own Phase 1 preparation is loaded.
    if saved["signature"] != manifest["tokenizer_signature"]:
        raise ValueError("Tokenizer signature mismatch.")
    tokenizer = A1BPETokenizer()
    tokenizer.__dict__.update(saved["state"])
    if tokenizer.vocab[:4] != [b"<pad>", b"<unk>", b"<s>", b"</s>"]:
        raise ValueError("Unexpected reserved token IDs.")
    probes = ["A court allowed the appeal.", "Dismissed.\nThe judgment follows.", "धारा १२"]
    for sample in probes:
        ids = tokenizer.encode(sample)
        if tokenizer.decode(ids) != sample or not ids or min(ids) < 4 or max(ids) >= len(tokenizer.vocab):
            raise ValueError("Saved BPE round-trip or token-ID range check failed.")
    return tokenizer, saved["signature"], manifest


def tokenize_documents(documents, tokenizer, cache_dir, identity):
    """Per-case atomic caches survive interruption; BPE is never retrained."""
    from functools import lru_cache
    folder = Path(cache_dir)
    folder.mkdir(parents=True, exist_ok=True)
    fingerprint = hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()
    @lru_cache(maxsize=250000)
    def piece_ids(piece):
        return tuple(tokenizer.encode(piece))
    last, started = time.monotonic(), time.monotonic()
    cache_files = []
    for i, doc in enumerate(documents):
        text_sha = hashlib.sha256(doc["text"].encode()).hexdigest()
        key = hashlib.sha256(json.dumps([fingerprint, doc["row_uid"], text_sha]).encode()).hexdigest()
        path = folder / (key + ".npz")
        if path.exists():
            with np.load(path, allow_pickle=False) as data:
                if str(data["identity"]) != key:
                    raise ValueError("Token-cache identity mismatch.")
                ids = data["ids"].copy()
        else:
            ids = np.asarray([t for piece in re.findall(r"\S+|\s+", doc["text"])
                              for t in piece_ids(piece)], dtype=np.int32)
            with open(str(path) + ".tmp", "wb") as f:
                np.savez_compressed(f, ids=ids, identity=np.asarray(key))
            Path(str(path) + ".tmp").replace(path)
        if ids.ndim != 1 or not len(ids) or ids.min() < 4 or ids.max() >= len(tokenizer.vocab):
            raise ValueError("Invalid cached content token IDs.")
        doc["token_ids"] = ids
        doc["text_sha256"] = text_sha
        doc["token_file"] = path.name
        cache_files.append({"id": doc["id"], "row_uid": doc["row_uid"], "row_index": doc["row_index"],
                            "split": doc["split"], "gold": doc["gold"],
                            "token_file": path.name, "text_sha256": text_sha, "content_tokens": len(ids)})
        if i == 0 or time.monotonic() - last >= 25 or i + 1 == len(documents):
            print("Tokenized/reused", i + 1, "/", len(documents), "cases |",
                  round((time.monotonic() - started) / 60, 1), "minutes", flush=True)
            last = time.monotonic()
    write_json(folder / "index.json", {"record_identity_version": 2, "fingerprint": fingerprint,
                                      "identity": identity, "cases": cache_files})
    return {"fingerprint": fingerprint, "cases": len(documents),
            "seconds": time.monotonic() - started, "unique_cached_pieces": piece_ids.cache_info().currsize}


def chunk_spans(n_tokens, total_tokens=512, overlap=64):
    content = total_tokens - 2
    if n_tokens < 1 or content < 1 or not 0 <= overlap < content:
        raise ValueError("Invalid chunk geometry.")
    spans = []
    start = 0
    while True:
        stop = min(start + content, n_tokens)
        spans.append((start, stop))
        if stop == n_tokens:
            return spans
        start += content - overlap


def selected_spans(n_tokens, total_tokens, overlap, max_chunks, policy="uniform"):
    spans = chunk_spans(n_tokens, total_tokens, overlap)
    if max_chunks < 1:
        raise ValueError("max_chunks must be positive.")
    if len(spans) <= max_chunks:
        return spans
    if policy == "uniform":
        indices = np.linspace(0, len(spans) - 1, max_chunks).round().astype(int).tolist()
    elif policy == "head":
        indices = list(range(max_chunks))
    elif policy == "tail":
        indices = list(range(len(spans) - max_chunks, len(spans)))
    elif policy == "head_tail":
        left = (max_chunks + 1) // 2
        indices = list(range(left)) + list(range(len(spans) - (max_chunks - left), len(spans)))
    else:
        raise ValueError("Unknown chunk selection policy.")
    if len(set(indices)) != max_chunks:
        raise AssertionError("Repeated selected chunk.")
    return [spans[i] for i in sorted(indices)]


def coverage(n_tokens, spans):
    covered, end = 0, 0
    for start, stop in spans:
        covered += max(0, stop - max(start, end))
        end = max(end, stop)
    return covered / n_tokens


def audit_documents(documents, schemas, output):
    import pandas as pd
    from sklearn.metrics import f1_score, precision_score, recall_score
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    rows, summary = [], {"schemas": schemas, "label_meaning": "Numeric mapping preserved; ACCEPT/REJECT meaning not assumed."}
    for split in ("train", "dev", "test"):
        part = [d for d in documents if d["split"] == split]
        counts = collections.Counter(d["gold"] for d in part) if split != "test" else {}
        summary[split] = {"cases": len(part), "labels": {str(k): v for k, v in counts.items()},
            "unique_original_case_ids": len({d["id"] for d in part}),
            "metric_and_sampling_unit": "provided rows; row_uid identifies each example",
            "content_tokens": {str(q): float(np.percentile([len(d["token_ids"]) for d in part], q))
                               for q in (0, 25, 50, 75, 90, 95, 99, 100)}}
        for d in part:
            rows.append({"split": split, "id": d["id"], "row_uid": d["row_uid"],
                "row_index": d["row_index"], "gold": d["gold"], "characters": len(d["text"]),
                "words": len(d["text"].split()), "content_tokens": len(d["token_ids"]),
                "text_sha256": d["text_sha256"]})
    problems, overlap = [], []
    for a, b in (("train", "dev"), ("train", "test"), ("dev", "test")):
        pa = [d for d in documents if d["split"] == a]
        pb = [d for d in documents if d["split"] == b]
        ids = sorted({d["id"] for d in pa} & {d["id"] for d in pb})
        # Whitespace-normalized duplicate detection catches formatting-only copies.
        def digest(d):
            return hashlib.sha256(re.sub(r"\s+", " ", d["text"]).strip().casefold().encode()).hexdigest()
        ma, mb = collections.defaultdict(list), collections.defaultdict(list)
        for d in pa:
            ma[digest(d)].append({"id": d["id"], "row_uid": d["row_uid"]})
        for d in pb:
            mb[digest(d)].append({"id": d["id"], "row_uid": d["row_uid"]})
        duplicate_pairs = [{"a_ids": ma[k], "b_ids": mb[k]} for k in sorted(set(ma) & set(mb))]
        overlap.append({"a": a, "b": b, "shared_ids": ids, "normalized_text_duplicates": duplicate_pairs})
        if ids or duplicate_pairs:
            problems.append("Overlap detected between " + a + " and " + b + "; inspect before training.")
    for split in ("train", "dev"):
        if set(summary[split]["labels"]) != {"0", "1"}:
            problems.append(split + " does not contain both binary classes.")
    summary["cross_split_overlap"] = overlap
    summary["training_blockers"] = problems
    within = {}
    for split in ("train", "dev", "test"):
        groups = collections.defaultdict(list)
        for d in documents:
            if d["split"] == split:
                groups[d["text_sha256"]].append({"id": d["id"], "row_uid": d["row_uid"], "gold": d["gold"]})
        within[split] = [ids for ids in groups.values() if len(ids) > 1]
    summary["within_split_exact_duplicate_groups"] = within
    majority = int(max(collections.Counter(d["gold"] for d in documents if d["split"] == "train").items(),
                       key=lambda pair: (pair[1], -pair[0]))[0])
    gold = [d["gold"] for d in documents if d["split"] == "dev"]
    summary["majority_label_from_train"] = majority
    summary["dev_constant_baselines"] = {}
    for label in (0, 1):
        pred = [label] * len(gold)
        summary["dev_constant_baselines"][str(label)] = {
            "macro_f1": float(f1_score(gold, pred, labels=[0, 1], average="macro", zero_division=0)),
            "precision_1": float(precision_score(gold, pred, zero_division=0)),
            "recall_1": float(recall_score(gold, pred, zero_division=0))}
    examples = []
    for split in ("train", "dev"):
        for label in (0, 1):
            candidates = [d for d in documents if d["split"] == split and d["gold"] == label]
            if candidates:
                d = sorted(candidates, key=lambda d: len(d["token_ids"]))[len(candidates) // 2]
                middle = len(d["text"]) // 2
                examples.append({"id": d["id"], "row_uid": d["row_uid"], "split": split, "gold": label,
                    "beginning": d["text"][:400], "middle": d["text"][middle:middle+400], "end": d["text"][-400:]})
    write_json(output / "data_audit.json", summary)
    write_json(output / "train_dev_examples.json", examples)
    pd.DataFrame(rows).to_csv(output / "case_statistics.csv", index=False)
    return summary


def audit_chunk_policies(documents, output, candidates):
    import pandas as pd
    rows = []
    for setting in candidates:
        for split in ("train", "dev", "test"):
            docs = [d for d in documents if d["split"] == split]
            spans = [selected_spans(len(d["token_ids"]), **setting) for d in docs]
            covered = [coverage(len(d["token_ids"]), s) for d, s in zip(docs, spans)]
            rows.append(dict(setting, split=split, cases=len(docs),
                mean_chunks=float(np.mean([len(s) for s in spans])),
                selected_chunks=sum(map(len, spans)), mean_content_coverage=float(np.mean(covered)),
                p10_content_coverage=float(np.percentile(covered, 10)),
                full_coverage_fraction=float(np.mean([v >= 1 - 1e-12 for v in covered])),
                total_encoded_tokens=sum(stop-start+2 for ss in spans for start, stop in ss)))
    pd.DataFrame(rows).to_csv(Path(output) / "chunk_policy_comparison.csv", index=False)
    return rows


def collate_cases(documents, total_tokens=512, overlap=64, max_chunks=8, policy="uniform"):
    import torch
    selected = [selected_spans(len(d["token_ids"]), total_tokens, overlap, max_chunks, policy)
                for d in documents]
    k = max(map(len, selected))
    width = max(stop-start+2 for spans in selected for start, stop in spans)
    ids = torch.zeros((len(documents), k, width), dtype=torch.long)
    lengths = torch.zeros((len(documents), k), dtype=torch.long)
    positions = torch.zeros((len(documents), k, 2), dtype=torch.float32)
    for b, (d, spans) in enumerate(zip(documents, selected)):
        for j, (start, stop) in enumerate(spans):
            tokens = [2] + [int(v) for v in d["token_ids"][start:stop]] + [3]
            ids[b, j, :len(tokens)] = torch.tensor(tokens)
            lengths[b, j] = len(tokens)
            positions[b, j] = torch.tensor([start / len(d["token_ids"]), stop / len(d["token_ids"])])
    labels = torch.tensor([d["gold"] if d["gold"] is not None else -100 for d in documents])
    return {"ids": ids, "lengths": lengths, "positions": positions}, labels

Overwriting /kaggle/working/cs781_phase4_prep_src/cjpe_data.py


In [9]:
%%writefile /kaggle/working/cs781_phase4_prep_src/cjpe_models.py
"""Phase 1-derived case classifiers. No external pretrained models or training loop."""
import contextlib
import copy
import math
from dataclasses import replace
import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from phase1_decoder import DecoderLM, LMConfig
import phase1_shared as shared
import phase1_efficient as efficient


def mask_for(lengths, width):
    return torch.arange(width, device=lengths.device)[None, :] < lengths[:, None]


def mean_content(states, lengths):
    positions = torch.arange(states.shape[1], device=states.device)[None, :]
    valid = (positions > 0) & (positions < lengths[:, None] - 1)
    count = valid.sum(1)
    if not bool((count > 0).all()):
        raise ValueError("Each chunk needs at least one content token.")
    return (states.float() * valid[:, :, None]).sum(1) / count[:, None]


def chunk_states(lm, architecture, ids, lengths, use_checkpoint=False):
    """Return token-context states and a full-context query/last-content readout."""
    if ids.ndim != 2 or lengths.shape != (len(ids),):
        raise ValueError("Expected IDs [chunks,tokens] and lengths [chunks].")
    if not bool(((lengths >= 3) & (lengths <= ids.shape[1])).all()):
        raise ValueError("Invalid nonpadding chunk length.")
    reserve = int(architecture in ("prefix", "encoder_decoder"))
    if ids.shape[1] + reserve > lm.cfg.max_seq_len:
        raise ValueError("Chunk plus classification query exceeds the Phase 1 context budget.")
    allowed = mask_for(lengths, ids.shape[1])[:, None, None, :]
    clean = ids.masked_fill(~allowed[:, 0, 0, :], 0)
    call = lambda block, *args: checkpoint(block, *args, use_reentrant=False,
                                            preserve_rng_state=True) if use_checkpoint else block(*args)
    if architecture == "decoder":
        x = lm.tok_emb(clean) * lm.cfg.embedding_scale + lm.pos_emb[:ids.shape[1]][None]
        for block in lm.blocks:
            x = call(block, x)
        states = lm.ln_f(x)
        readout = states[torch.arange(len(ids), device=ids.device), lengths - 2]
    elif architecture == "efficient":
        x = lm.embed(clean)
        for block in lm.blocks:
            x = call(block, x, lengths)
        states = lm.final_norm(x)
        readout = states[torch.arange(len(ids), device=ids.device), lengths - 2]
    elif architecture == "encoder_decoder":
        memory = lm.embed(clean)
        for block in lm.encoder:
            memory = call(block, memory, allowed)
        states = lm.encoder_norm(memory)
        query = torch.full((len(ids), 1), 2, dtype=torch.long, device=ids.device)
        x = lm.embed(query)
        for block in lm.decoder:
            x = call(block, x, states, allowed)
        readout = lm.decoder_norm(x)[:, 0]
    elif architecture == "prefix":
        p = lm.embed(clean, role=0)
        query = torch.full((len(ids), 1), 2, dtype=torch.long, device=ids.device)
        d = lm.embed(query, offset=lengths, role=1)
        for block in lm.blocks:
            p, d = call(block, p, d, allowed)
        states, readout = lm.final_norm(p), lm.final_norm(d)[:, 0]
    else:
        raise ValueError("Unknown architecture.")
    return states, readout


class Bottleneck(nn.Module):
    def __init__(self, width, bottleneck, dropout):
        super().__init__()
        self.down = nn.Linear(width, bottleneck)
        self.up = nn.Linear(bottleneck, width)
        self.dropout = nn.Dropout(dropout)
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, x):
        return x + self.dropout(self.up(F.gelu(self.down(x))))


class AdaptedOutput(nn.Module):
    """Adapter after a sublayer output, before its outer residual addition."""
    def __init__(self, original, width, bottleneck, dropout):
        super().__init__()
        self.original = original
        self.adapter = Bottleneck(width, bottleneck, dropout)

    def forward(self, *args, **kwargs):
        return self.adapter(self.original(*args, **kwargs))


class LoRMAPlusLinear(nn.Module):
    """Output-side (I + alpha/r BA) W0 x + original bias."""
    def __init__(self, original, rank, alpha, dropout):
        super().__init__()
        if not isinstance(original, nn.Linear) or not 0 < rank <= min(original.weight.shape):
            raise ValueError("Invalid LoRMA+ linear target or rank.")
        self.original = original
        self.lorma_A = nn.Parameter(original.weight.new_empty(rank, original.out_features))
        self.lorma_B = nn.Parameter(original.weight.new_zeros(original.out_features, rank))
        nn.init.kaiming_uniform_(self.lorma_A, a=math.sqrt(5))
        self.dropout = nn.Dropout(dropout)
        self.scaling = alpha / rank
        for p in original.parameters():
            p.requires_grad_(False)

    def forward(self, x):
        base = self.original(x)
        projected = base if self.original.bias is None else base - self.original.bias.to(base.dtype)
        return base + self.scaling * F.linear(F.linear(self.dropout(projected), self.lorma_A), self.lorma_B)


class DocumentBiGRU(nn.Module):
    """Small GRU written with tensor operations, applied only to valid chunks."""
    def __init__(self, width, hidden):
        super().__init__()
        self.hidden = hidden
        self.input = nn.ModuleList([nn.Linear(width, 3 * hidden) for _ in range(2)])
        self.recurrent = nn.ModuleList([nn.Linear(hidden, 3 * hidden) for _ in range(2)])

    def forward(self, x):
        outputs = []
        for direction in range(2):
            hidden = x.new_zeros(self.hidden)
            transformed = self.input[direction](x)
            states = [None] * len(x)
            order = range(len(x)) if direction == 0 else range(len(x)-1, -1, -1)
            for i in order:
                xr, xz, xn = transformed[i].chunk(3, -1)
                hr, hz, hn = self.recurrent[direction](hidden).chunk(3, -1)
                reset, update = torch.sigmoid(xr+hr), torch.sigmoid(xz+hz)
                candidate = torch.tanh(xn + reset*hn)
                hidden = (1-update)*candidate + update*hidden
                states[i] = hidden
            outputs.append(torch.stack(states))
        return torch.cat(outputs, -1)


def compact_efficient(lm, layers):
    if not layers or len(set(layers)) != len(layers) or list(layers) != sorted(layers):
        raise ValueError("Student layers must be distinct, ascending zero-based indices.")
    if min(layers) < 0 or max(layers) >= len(lm.blocks) or len(layers) >= len(lm.blocks):
        raise ValueError("The student must retain fewer than all original efficient blocks.")
    result = efficient.EfficientLM(replace(lm.cfg, n_layers=len(layers)))
    result.tok_emb.load_state_dict(lm.tok_emb.state_dict(), strict=True)
    result.final_norm.load_state_dict(lm.final_norm.state_dict(), strict=True)
    for new, old in enumerate(layers):
        result.blocks[new].load_state_dict(lm.blocks[old].state_dict(), strict=True)
    if result.tok_emb.weight.data_ptr() == lm.tok_emb.weight.data_ptr():
        raise AssertionError("Student and original must not share mutable parameter storage.")
    return result


def configure_dropout(lm, probability):
    lm.cfg = replace(lm.cfg, dropout=float(probability))
    for module in lm.modules():
        if isinstance(module, nn.Dropout):
            module.p = float(probability)
        elif isinstance(module, (shared.Attention, efficient.Attention)):
            module.dropout = float(probability)


class CaseClassifier(nn.Module):
    def __init__(self, lm, architecture, settings):
        super().__init__()
        self.lm, self.architecture, self.settings = lm, architecture, dict(settings)
        self.use_amp = True
        self.gradient_checkpointing = bool(settings["gradient_checkpointing"])
        self.chunk_microbatch = int(settings["chunk_microbatch"])
        configure_dropout(lm, settings["backbone_dropout"])
        for p in lm.parameters():
            p.requires_grad_(False)
        self.adaptation_targets = []
        if architecture == "decoder":
            upper = settings["decoder_upper_layers"]
            if not 1 <= upper <= len(lm.blocks):
                raise ValueError("Invalid partial fine-tuning depth.")
            for i in range(len(lm.blocks)-upper, len(lm.blocks)):
                for p in lm.blocks[i].parameters():
                    p.requires_grad_(True)
                self.adaptation_targets.append("blocks." + str(i))
            for p in lm.ln_f.parameters():
                p.requires_grad_(True)
        elif architecture == "encoder_decoder":
            for group, blocks in (("encoder", lm.encoder), ("decoder", lm.decoder)):
                for i, block in enumerate(blocks):
                    names = ("attn", "ffn") if group == "encoder" else ("self_attn", "cross_attn", "ffn")
                    for name in names:
                        setattr(block, name, AdaptedOutput(getattr(block, name), lm.cfg.d_model,
                            settings["adapter_bottleneck"], settings["adapter_dropout"]))
                        self.adaptation_targets.append(group + "." + str(i) + "." + name)
            if settings["adapter_train_layer_norm"]:
                for module in lm.modules():
                    if isinstance(module, nn.LayerNorm):
                        for p in module.parameters():
                            p.requires_grad_(True)
        elif architecture == "prefix":
            for i, block in enumerate(lm.blocks):
                for name in ("q_proj", "k_proj", "v_proj", "out_proj"):
                    setattr(block.attn, name, LoRMAPlusLinear(getattr(block.attn, name),
                        settings["lorma_rank"], settings["lorma_alpha"], settings["lorma_dropout"]))
                    self.adaptation_targets.append("blocks." + str(i) + ".attn." + name)
                for j in (0, 2):
                    block.ffn.net[j] = LoRMAPlusLinear(block.ffn.net[j], settings["lorma_rank"],
                        settings["lorma_alpha"], settings["lorma_dropout"])
                    self.adaptation_targets.append("blocks." + str(i) + ".ffn.net." + str(j))
        elif architecture == "efficient":
            for p in lm.parameters():
                p.requires_grad_(True)
        else:
            raise ValueError("Unknown classifier architecture.")
        # Seed the task head separately: matched gold and KD students get identical heads.
        with torch.random.fork_rng(devices=[]):
            torch.manual_seed(settings["head_seed"])
            self.projection = nn.Linear(2*lm.cfg.d_model + 2, settings["projection_dim"])
            self.context = DocumentBiGRU(settings["projection_dim"], settings["context_hidden"])
            width = 2*settings["context_hidden"]
            self.attention = nn.Sequential(nn.Linear(width, settings["attention_hidden"]), nn.Tanh(),
                                          nn.Linear(settings["attention_hidden"], 1, bias=False))
            self.classifier = nn.Linear(width, 2)
        self.dropout = nn.Dropout(settings["head_dropout"])

    def chunk_vectors(self, ids, lengths):
        states, readout = chunk_states(self.lm, self.architecture, ids, lengths,
            self.training and self.gradient_checkpointing)
        return torch.cat([mean_content(states, lengths), readout.float()], -1)

    def forward(self, ids, lengths, positions):
        if ids.ndim != 3 or lengths.shape != ids.shape[:2] or positions.shape != (*ids.shape[:2], 2):
            raise ValueError("Expected one chunk sequence per case.")
        logits, weights = [], []
        with torch.autocast("cuda", dtype=torch.float16, enabled=ids.is_cuda and self.use_amp):
            for b in range(len(ids)):
                active = lengths[b] > 0
                n = int(active.sum())
                if n == 0 or not bool(active[:n].all()) or bool(active[n:].any()):
                    raise ValueError("Valid chunks must form one nonempty prefix.")
                pieces = []
                for start in range(0, n, self.chunk_microbatch):
                    stop = min(start+self.chunk_microbatch, n)
                    width = int(lengths[b, start:stop].max())
                    pieces.append(self.chunk_vectors(ids[b, start:stop, :width], lengths[b, start:stop]))
                chunk_vectors = torch.cat(pieces)
                projected = torch.tanh(self.projection(torch.cat([chunk_vectors, positions[b, :n].float()], -1)))
                contextual = self.context(self.dropout(projected))
                alpha = torch.softmax(self.attention(contextual).squeeze(-1).float(), 0)
                document = (contextual.float()*alpha[:, None]).sum(0)
                logits.append(self.classifier(self.dropout(document)))
                weights.append(F.pad(alpha, (0, ids.shape[1]-n)))
        return {"logits": torch.stack(logits).float(), "chunk_attention": torch.stack(weights),
                "device_ids": ids.new_full((len(ids),), ids.device.index if ids.is_cuda else -1)}


def kd_objective(student_logits, labels, teacher_logits=None, temperature=2.0, kd_weight=0.5):
    if temperature <= 0 or not 0 <= kd_weight <= 1:
        raise ValueError("Invalid distillation temperature or mixing weight.")
    if bool(((labels < 0) | (labels > 1)).any()):
        raise ValueError("Training loss requires labelled training cases.")
    gold = F.cross_entropy(student_logits.float(), labels)
    kd = gold*0
    if teacher_logits is not None:
        if teacher_logits.shape != student_logits.shape:
            raise ValueError("Teacher/student class-order or batch shape mismatch.")
        q = torch.softmax(teacher_logits.detach().float()/temperature, -1)
        kd = F.kl_div(F.log_softmax(student_logits.float()/temperature, -1), q,
                      reduction="batchmean") * temperature**2
    elif kd_weight:
        raise ValueError("KD training requires genuine trained-teacher logits.")
    return (1-kd_weight)*gold + kd_weight*kd, {"gold_ce": gold.detach(), "kd": kd.detach()}


def expected_trainability(model, name):
    if not name.startswith("lm."):
        return True
    if model.architecture == "efficient":
        return True
    if model.architecture == "decoder":
        upper = len(model.lm.blocks)-model.settings["decoder_upper_layers"]
        return name.startswith("lm.ln_f.") or any(name.startswith("lm.blocks."+str(i)+".")
                                                  for i in range(upper, len(model.lm.blocks)))
    if model.architecture == "prefix":
        return name.endswith((".lorma_A", ".lorma_B"))
    if model.architecture == "encoder_decoder":
        if ".adapter." in name:
            return True
        if model.settings["adapter_train_layer_norm"]:
            return any(name.startswith(prefix) for prefix in (
                "lm.encoder_norm.", "lm.decoder_norm.")) or any(
                ".ln"+str(i)+"." in name for i in (1, 2, 3))
        return False
    raise ValueError("Unknown architecture.")


def trainability_audit(model):
    counts = {"total_parameters": 0, "trainable_parameters": 0, "backbone_trainable": 0, "task_trainable": 0}
    for name, p in model.named_parameters():
        if p.requires_grad != expected_trainability(model, name):
            raise AssertionError("Unexpected trainability: " + name)
        counts["total_parameters"] += p.numel()
        if p.requires_grad:
            counts["trainable_parameters"] += p.numel()
            counts["backbone_trainable" if name.startswith("lm.") else "task_trainable"] += p.numel()
    counts["adaptation_targets"] = model.adaptation_targets
    counts["parameter_bytes_float32"] = counts["total_parameters"]*4
    return counts

Overwriting /kaggle/working/cs781_phase4_prep_src/cjpe_models.py


In [10]:
%%writefile /kaggle/working/cs781_phase4_prep_src/cjpe_checks.py
"""Bounded correctness, temporary learning, two-device and resource probes."""
import copy
import gc
import hashlib
import io
import math
import random
import time
from dataclasses import asdict
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from cjpe_data import sha256, write_json, safe_error, collate_cases, chunk_spans, selected_spans, coverage
from cjpe_models import (CaseClassifier, Bottleneck, LoRMAPlusLinear, DocumentBiGRU,
    compact_efficient, chunk_states, kd_objective, trainability_audit)
from phase1_decoder import DecoderLM, LMConfig
import phase1_shared as shared
import phase1_efficient as efficient


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def tensor_hash(tensor):
    return hashlib.sha256(tensor.detach().cpu().contiguous().numpy().tobytes()).hexdigest()


def trainable_hash(model):
    h = hashlib.sha256()
    for name, p in model.named_parameters():
        if p.requires_grad:
            h.update(name.encode())
            h.update(p.detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


def load_base(path, architecture, signature):
    saved = torch.load(path, map_location="cpu", weights_only=False)
    if saved["tokenizer_signature"] != signature:
        raise ValueError("Phase 1 checkpoint and prep-2 tokenizer signatures differ.")
    cfg = saved["model_config"]
    if architecture == "decoder":
        lm = DecoderLM(LMConfig(**cfg))
    elif architecture == "efficient":
        lm = efficient.EfficientLM(efficient.ModelConfig(**cfg))
    else:
        config = shared.ModelConfig(**cfg)
        if config.architecture != architecture:
            raise ValueError("Incorrect checkpoint architecture.")
        lm = shared.EncoderDecoderLM(config) if architecture == "encoder_decoder" else shared.PrefixLM(config)
    if not torch.equal(saved["model_state"]["tok_emb.weight"], saved["model_state"]["head.weight"]):
        raise ValueError("The checkpoint's tied embedding/LM head values disagree.")
    lm.load_state_dict(saved["model_state"], strict=True)
    if lm.head.weight is not lm.tok_emb.weight:
        raise AssertionError("Original embedding/output weight tying was lost.")
    report = {"model_config": cfg, "checkpoint_keys": sorted(saved),
              "original_parameters": sum(p.numel() for p in lm.parameters()),
              "strict_state_dict_load": True, "tied_embedding_preserved": True,
              "tokenizer_signature_matches": True}
    del saved
    gc.collect()
    return lm, report


def native_readout_check(lm, architecture, device):
    lm = lm.to(device).eval()
    width = min(17, lm.cfg.max_seq_len-1)
    ids = torch.randint(4, lm.cfg.vocab_size, (2, width), device=device)
    ids[:, 0], ids[:, -1] = 2, 3
    lengths = torch.full((2,), width, device=device, dtype=torch.long)
    query = torch.full((2, 1), 2, device=device, dtype=torch.long)
    with torch.no_grad():
        _, hidden = chunk_states(lm, architecture, ids, lengths)
        if architecture in ("prefix", "encoder_decoder"):
            expected = lm(ids, query, lengths)[:, 0]
        else:
            expected = lm(ids)[:, -2]
        actual = lm.head(hidden)
    torch.testing.assert_close(actual, expected, atol=2e-5, rtol=2e-4)
    return {"status": "passed", "max_abs_logit_difference": float((actual-expected).abs().max()),
            "query": "single BOS token through original decoder stream" if architecture in
            ("prefix", "encoder_decoder") else "last content token"}


def helper_checks():
    seed_all(19)
    for n in (1, 17, 30, 31, 32, 100, 1001):
        all_spans = chunk_spans(n, 32, 7)
        assert coverage(n, all_spans) == 1.0
        assert all(0 <= a < b <= n and b-a <= 30 for a, b in all_spans)
        for cap in (1, 2, 4, 8):
            for policy in ("uniform", "head", "tail", "head_tail"):
                spans = selected_spans(n, 32, 7, cap, policy)
                assert len(spans) <= cap and len(set(spans)) == len(spans)
                assert 0 < coverage(n, spans) <= 1.0
    original = nn.Linear(5, 7, bias=True).double()
    adapter = LoRMAPlusLinear(original, 2, 4, 0).double()
    x = torch.randn(3, 5, dtype=torch.float64, requires_grad=True)
    torch.testing.assert_close(adapter(x), original(x), rtol=0, atol=0)
    with torch.no_grad():
        adapter.lorma_B.normal_(std=0.05)
    matrix = original.weight + adapter.scaling*adapter.lorma_B@adapter.lorma_A@original.weight
    torch.testing.assert_close(adapter(x), F.linear(x, matrix, original.bias), atol=1e-12, rtol=1e-12)
    assert torch.autograd.gradcheck(adapter, (x,), eps=1e-6, atol=1e-5)
    bottleneck = Bottleneck(7, 3, 0).double()
    y = torch.randn(4, 7, dtype=torch.float64)
    torch.testing.assert_close(bottleneck(y), y, atol=0, rtol=0)
    # Compare the manually implemented GRU against an independent library reference.
    manual = DocumentBiGRU(5, 3).double()
    reference = nn.GRU(5, 3, bidirectional=True).double()
    with torch.no_grad():
        for direction, suffix in ((0, ""), (1, "_reverse")):
            getattr(reference, "weight_ih_l0"+suffix).copy_(manual.input[direction].weight)
            getattr(reference, "bias_ih_l0"+suffix).copy_(manual.input[direction].bias)
            getattr(reference, "weight_hh_l0"+suffix).copy_(manual.recurrent[direction].weight)
            getattr(reference, "bias_hh_l0"+suffix).copy_(manual.recurrent[direction].bias)
    seq = torch.randn(6, 5, dtype=torch.float64)
    torch.testing.assert_close(manual(seq), reference(seq[:, None])[0][:, 0], atol=1e-12, rtol=1e-12)
    logits = torch.tensor([[0.3, -0.2], [-0.7, 1.0]], requires_grad=True)
    labels = torch.tensor([0, 1])
    total, pieces = kd_objective(logits, labels, logits.detach(), temperature=2, kd_weight=.5)
    assert abs(float(pieces["kd"])) < 1e-6
    gold, _ = kd_objective(logits, labels, kd_weight=0)
    torch.testing.assert_close(gold, F.cross_entropy(logits, labels))
    total.backward()
    assert torch.isfinite(logits.grad).all()
    try:
        kd_objective(logits, labels)
    except ValueError:
        pass
    else:
        raise AssertionError("KD without teacher targets must be rejected.")
    return {"status": "passed", "checks": ["chunk bounds and full-coverage geometry",
        "LoRMA+ identity, rectangular biased matrix equivalence and numerical input gradients",
        "zero-initialized bottleneck identity", "manual BiGRU versus nn.GRU",
        "binary CE/KD values, gradients and missing-teacher guard"],
        "synthetic_kd_targets_are_not_valid_for_real_training": True}


def batch_to(batch, device):
    return {k: v.to(device) for k, v in batch.items()}


def padding_check(model, batch):
    model.eval()
    old = model.use_amp
    model.use_amp = False
    with torch.no_grad():
        original = model(**batch)
        changed = {k: v.clone() for k, v in batch.items()}
        valid_tokens = torch.arange(batch["ids"].shape[2], device=batch["ids"].device)[None, None] < batch["lengths"][:, :, None]
        changed["ids"][~valid_tokens] = 5
        altered = model(**changed)
        extended = {
            "ids": F.pad(batch["ids"], (0, 3, 0, 2), value=9),
            "lengths": F.pad(batch["lengths"], (0, 2)),
            "positions": F.pad(batch["positions"], (0, 0, 0, 2), value=0.7)}
        padded = model(**extended)
    torch.testing.assert_close(original["logits"], altered["logits"], atol=3e-5, rtol=3e-4)
    torch.testing.assert_close(original["logits"], padded["logits"], atol=3e-5, rtol=3e-4)
    torch.testing.assert_close(original["chunk_attention"].sum(1),
                               torch.ones(len(batch["ids"]), device=batch["ids"].device))
    assert bool((padded["chunk_attention"][:, -2:] == 0).all())
    assert torch.isfinite(original["logits"]).all()
    model.use_amp = old
    return {"status": "passed", "changed_pad_tokens_and_extra_pad_chunks_ignored": True,
            "attention_weights_normalize_over_real_chunks": True}


def two_gpu_check(model, batch, labels):
    if not batch["ids"].is_cuda or torch.cuda.device_count() < 2:
        return {"status": "not_run", "reason": "Two CUDA devices unavailable."}
    model.eval()
    old = model.use_amp
    model.use_amp = False
    model.zero_grad(set_to_none=True)
    serial = model(**batch)["logits"]
    F.cross_entropy(serial, labels).backward()
    gradients = {n: p.grad.detach().cpu().clone() for n, p in model.named_parameters()
                 if p.requires_grad and p.grad is not None}
    model.zero_grad(set_to_none=True)
    parallel = nn.DataParallel(model, device_ids=[0, 1])
    result = parallel(**batch)
    assert set(result["device_ids"].cpu().tolist()) == {0, 1}
    F.cross_entropy(result["logits"], labels).backward()
    torch.testing.assert_close(serial, result["logits"], rtol=1e-3, atol=3e-5)
    maximum = 0.0
    for name, p in model.named_parameters():
        if name in gradients:
            if p.grad is None:
                raise AssertionError("Missing parallel gradient: " + name)
            actual = p.grad.detach().cpu()
            torch.testing.assert_close(actual, gradients[name], rtol=2e-3, atol=5e-5)
            maximum = max(maximum, float((actual-gradients[name]).abs().max()))
        elif p.requires_grad and p.grad is not None and float(p.grad.abs().max()) > 0:
            raise AssertionError("Unexpected additional parallel gradient: " + name)
    model.zero_grad(set_to_none=True)
    model.use_amp = old
    del parallel, gradients
    return {"status": "passed", "devices_observed": [0, 1],
            "max_abs_gradient_difference": maximum, "dropout_disabled_for_comparison": True}


def temporary_learning(model, batch, labels, steps):
    device = batch["ids"].device
    model.use_amp = device.type == "cuda"
    # Isolate gradient/learning mechanics from stochastic regularization in this tiny test.
    dropout_backup = [(m, "p", m.p) for m in model.modules() if isinstance(m, nn.Dropout)]
    dropout_backup += [(m, "dropout", m.dropout) for m in model.modules()
                       if isinstance(getattr(m, "dropout", None), float)]
    for module, attribute, _ in dropout_backup:
        setattr(module, attribute, 0.0)
    def evaluation_loss():
        model.eval()
        with torch.no_grad():
            return float(F.cross_entropy(model(**batch)["logits"], labels))
    before = evaluation_loss()
    optimizer = torch.optim.AdamW([
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and n.startswith("lm.")],
         "lr": 1e-5 if model.architecture in ("decoder", "efficient") else 1e-4},
        {"params": [p for n, p in model.named_parameters() if p.requires_grad and not n.startswith("lm.")],
         "lr": 1e-3}], weight_decay=0.0)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    rows, started = [], time.monotonic()
    gradient_audit = {}
    for step in range(steps):
        model.train()
        optimizer.zero_grad(set_to_none=True)
        logits = model(**batch)["logits"]
        # KD wiring is checked in the student too; these synthetic targets never persist.
        teacher = torch.stack([1.5*(labels == 0), 1.5*(labels == 1)], -1).float() if model.architecture == "efficient" else None
        loss, _ = kd_objective(logits, labels, teacher, kd_weight=.3 if teacher is not None else 0)
        if not torch.isfinite(loss):
            raise ValueError("Nonfinite temporary training loss.")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        grads = [p.grad for p in model.parameters() if p.requires_grad and p.grad is not None]
        if not grads or not all(bool(torch.isfinite(g).all()) for g in grads):
            raise ValueError("Nonfinite or missing temporary gradients.")
        frozen_bad = [n for n, p in model.named_parameters() if not p.requires_grad and p.grad is not None]
        if frozen_bad:
            raise AssertionError("Frozen weights received gradients.")
        if step in (0, steps-1):
            group_norms = {}
            for group, prefix in (("backbone_or_adaptation", "lm."), ("task_components", None)):
                square_sum = sum(float(p.grad.detach().float().square().sum())
                    for n, p in model.named_parameters() if p.requires_grad and p.grad is not None
                    and (n.startswith(prefix) if prefix else not n.startswith("lm.")))
                group_norms[group] = math.sqrt(square_sum)
            if not all(v > 0 and math.isfinite(v) for v in group_norms.values()):
                raise AssertionError("A required learning component has zero or nonfinite gradients.")
            gradient_audit[str(step+1)] = group_norms
        torch.nn.utils.clip_grad_norm_([p for p in model.parameters() if p.requires_grad], 1.)
        scaler.step(optimizer)
        scaler.update()
        rows.append({"step": step+1, "loss": float(loss.detach())})
    after = evaluation_loss()
    model.zero_grad(set_to_none=True)
    del optimizer, scaler
    for module, attribute, probability in dropout_backup:
        setattr(module, attribute, probability)
    return {"status": "passed" if after < before-1e-5 else "needs_review",
            "before_gold_ce": before, "after_gold_ce": after, "steps": steps,
            "seconds": time.monotonic()-started, "curve": rows,
            "gradient_group_norms": gradient_audit,
            "scope": "two training cases, short chunks; wiring/memorization only",
            "temporary_dropout_disabled_and_restored": True,
            "student_teacher_targets": "synthetic_wiring_only" if model.architecture == "efficient" else None}


def checkpoint_roundtrip(model, temporary_dir):
    path = Path(temporary_dir) / "temporary_trainable_roundtrip.pt"
    state = {n: p.detach().cpu().clone() for n, p in model.named_parameters() if p.requires_grad}
    torch.save({"trainable": state, "architecture": model.architecture}, path)
    expected = trainable_hash(model)
    with torch.no_grad():
        next(p for p in model.parameters() if p.requires_grad).add_(1)
        saved = torch.load(path, map_location="cpu", weights_only=True)
        for name, p in model.named_parameters():
            if p.requires_grad:
                p.copy_(saved["trainable"][name])
    if trainable_hash(model) != expected:
        raise AssertionError("Trainable state checkpoint round trip failed.")
    path.unlink()
    return {"status": "passed", "checkpoint_scope": "trainable weights plus original Phase 1 dependency",
            "optimizer_resume_check": "not performed; full trainer is not part of preparation"}


def resource_probes(model, train_documents, settings, device):
    """Stress maximum chunks; successful probes include allocated AdamW state."""
    candidates = settings["resource_candidates"]
    results = []
    parallel = nn.DataParallel(model, device_ids=[0, 1]) if device.type == "cuda" and torch.cuda.device_count() >= 2 else model
    model.train()
    model.use_amp = device.type == "cuda"
    overall_started = time.monotonic()
    for candidate in candidates:
        if time.monotonic()-overall_started > settings.get("resource_budget_minutes", 12)*60:
            results.append(dict(candidate, status="skipped_budget", reason="Per-architecture resource budget reached."))
            continue
        print("Resource probe:", model.architecture, candidate, flush=True)
        optimizer = scaler = batch = labels = output = loss = None
        try:
            width, count, micro = candidate["total_tokens"], candidate["max_chunks"], candidate["chunk_microbatch"]
            model.chunk_microbatch = micro
            # Controlled full-length stress cases; no labels or tokens from development/test.
            source = train_documents[0]["token_ids"]
            required = (width-2)*count
            repeated = np.resize(source, required).astype(np.int32)
            stress = [{"token_ids": repeated, "gold": i, "id": "stress_"+str(i)} for i in (0, 1)]
            batch, labels = collate_cases(stress, total_tokens=width, overlap=0, max_chunks=count)
            batch, labels = batch_to(batch, device), labels.to(device)
            if device.type == "cuda":
                for i in range(min(2, torch.cuda.device_count())):
                    torch.cuda.synchronize(i)
                    torch.cuda.reset_peak_memory_stats(i)
            optimizer = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=0.0)
            scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
            started = time.monotonic()
            samples = []
            for step in range(2):
                optimizer.zero_grad(set_to_none=True)
                one = time.monotonic()
                output = parallel(**batch)
                loss = F.cross_entropy(output["logits"], labels)
                if not torch.isfinite(loss):
                    raise ValueError("Nonfinite resource-probe loss.")
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                if any(p.grad is not None and not bool(torch.isfinite(p.grad).all()) for p in model.parameters()):
                    raise ValueError("Nonfinite resource-probe gradients.")
                scaler.step(optimizer)
                scaler.update()
                if device.type == "cuda":
                    torch.cuda.synchronize(0)
                    if torch.cuda.device_count() > 1:
                        torch.cuda.synchronize(1)
                samples.append(time.monotonic()-one)
            devices = sorted(set(output["device_ids"].detach().cpu().tolist()))
            memory = [{"device": i, "peak_allocated_gib": torch.cuda.max_memory_allocated(i)/2**30,
                       "peak_reserved_gib": torch.cuda.max_memory_reserved(i)/2**30,
                       "total_gib": torch.cuda.get_device_properties(i).total_memory/2**30}
                      for i in range(torch.cuda.device_count())] if device.type == "cuda" else []
            results.append(dict(candidate, status="passed", seconds_per_two_cases=samples[-1],
                cold_step_seconds=samples[0], devices_observed=devices, memory=memory,
                allocated_adamw_state=True, zero_lr_preserves_parameters=True,
                scope="maximum chunks and length, one case per GPU, two forward/backward/AdamW steps"))
        except torch.cuda.OutOfMemoryError:
            results.append(dict(candidate, status="oom", reason="CUDA out of memory; smaller candidates still tested."))
        except Exception as exc:
            results.append(dict(candidate, status="failed", reason=safe_error(exc)))
        finally:
            model.zero_grad(set_to_none=True)
            del optimizer, scaler, batch, labels, output, loss
            gc.collect()
            if device.type == "cuda":
                torch.cuda.empty_cache()
    del parallel
    model.chunk_microbatch = settings["chunk_microbatch"]
    return results


def architecture_probe(architecture, checkpoint_path, signature, settings, documents, output,
                       device=torch.device("cuda:0")):
    output = Path(output)
    output.mkdir(parents=True, exist_ok=True)
    model = lm = None
    report = {"architecture": architecture, "status": "running"}
    try:
        seed_all(settings["seed"])
        lm, identity = load_base(checkpoint_path, architecture, signature)
        report["phase1"] = identity
        if architecture == "efficient":
            old_parameters = identity["original_parameters"]
            original = lm
            lm = compact_efficient(original, settings["student_layers"])
            # All copied values are compared, including every retained block parameter.
            for new, old in enumerate(settings["student_layers"]):
                for name, tensor in lm.blocks[new].state_dict().items():
                    if not torch.equal(tensor, original.blocks[old].state_dict()[name]):
                        raise AssertionError("Student block inheritance mismatch.")
            if not torch.equal(lm.tok_emb.weight, original.tok_emb.weight):
                raise AssertionError("Student embedding inheritance mismatch.")
            report["student"] = {"retained_zero_based_layers": settings["student_layers"],
                "original_backbone_parameters": old_parameters,
                "student_backbone_parameters": sum(p.numel() for p in lm.parameters()),
                "copied_block_and_embedding_values_verified": True}
            del original
            gc.collect()
        report["native_readout"] = native_readout_check(lm, architecture, device)
        # Compare initialized PEFT paths against the unadapted readout before learning.
        example_ids = torch.tensor([[2, 9, 12, 7, 3]], device=device)
        example_lengths = torch.tensor([5], device=device)
        with torch.no_grad():
            before = chunk_states(lm, architecture, example_ids, example_lengths)
        model = CaseClassifier(lm, architecture, settings).to(device)
        lm = None
        model.eval()
        with torch.no_grad():
            after = chunk_states(model.lm, architecture, example_ids, example_lengths)
        for a, b in zip(before, after):
            torch.testing.assert_close(a, b, rtol=0, atol=0)
        report["adaptation_initial_identity"] = "passed"
        report["parameters"] = trainability_audit(model)
        report["initial_trainable_sha256"] = trainable_hash(model)
        train_docs = [d for d in documents if d["split"] == "train"]
        tiny_docs = [min((d for d in train_docs if d["gold"] == label),
                         key=lambda d: len(d["token_ids"])) for label in (0, 1)]
        # Different valid lengths/chunk counts exercise masking.
        tiny_docs = [dict(d, token_ids=d["token_ids"][:(35 if i == 0 else 93)]) for i, d in enumerate(tiny_docs)]
        batch, labels = collate_cases(tiny_docs, total_tokens=48, overlap=4, max_chunks=2)
        batch, labels = batch_to(batch, device), labels.to(device)
        report["padding_and_pooling"] = padding_check(model, batch)
        report["two_gpu"] = two_gpu_check(model, batch, labels)
        # Temporary optimizer updates are isolated to this disposable model instance.
        report["temporary_learning"] = temporary_learning(model, batch, labels, settings["smoke_steps"])
        report["checkpoint_roundtrip"] = checkpoint_roundtrip(model, output)
        del batch, labels, before, after
        if settings["run_resource_probes"]:
            report["resources"] = resource_probes(model, train_docs, settings, device)
        else:
            report["resources"] = []
        passed = [r for r in report["resources"] if r["status"] == "passed"]
        good_margin = [r for r in passed if not r["memory"] or
                       max(m["peak_allocated_gib"]/m["total_gib"] for m in r["memory"]) < .85]
        choice = max(good_margin or passed, key=lambda r: (r["total_tokens"], r["max_chunks"])) if passed else None
        report["recommended_resource_configuration"] = choice
        blockers = []
        if report["temporary_learning"]["status"] != "passed":
            blockers.append("Tiny training-set loss did not improve; inspect the curve.")
        if settings["require_two_gpus"] and report["two_gpu"]["status"] != "passed":
            blockers.append("Required two-GPU gradient check was not completed.")
        if settings["run_resource_probes"] and not passed:
            blockers.append("No resource configuration passed.")
        if choice:
            count = len(train_docs)
            report["runtime_projection"] = {
                "stress_based_train_epoch_minutes": choice["seconds_per_two_cases"]*math.ceil(count/2)/60,
                "assumptions": "One case/GPU; padded maximum-chunk stress workload. Excludes validation, exports and I/O.",
                "not_a_measured_full_epoch": True}
        report["training_blockers"] = blockers
        report["status"] = "passed" if not blockers else "needs_review"
        report["temporary_models_discarded"] = True
        report["real_teacher_cache_created"] = False
    except KeyboardInterrupt:
        report.update(status="interrupted", error="Stopped by user; partial reports retained.")
        raise
    except Exception as exc:
        report.update(status="failed", error_type=type(exc).__name__, error=safe_error(exc))
    finally:
        if model is not None:
            model.zero_grad(set_to_none=True)
        del model, lm
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        write_json(output / "probe_report.json", report)
    return report

Overwriting /kaggle/working/cs781_phase4_prep_src/cjpe_checks.py


In [11]:
%%writefile /kaggle/working/cs781_phase4_prep_src/cjpe_reuse.py
"""Load the prepared training inputs in a later notebook without retokenizing."""
from pathlib import Path
import numpy as np
from cjpe_data import read_json, sha256


def load_prepared_cases(prepared_root):
    root = Path(prepared_root)
    if not (root/"token_cache"/"index.json").exists():
        choices = list(root.rglob("token_cache/index.json"))
        if len(choices) != 1:
            raise ValueError("Choose the extracted phase4_cjpe_prepared folder.")
        root = choices[0].parent.parent
    cache = root/"token_cache"
    index = read_json(cache/"index.json")
    if index.get("record_identity_version") != 2:
        raise ValueError("This older cache lacks unique row identities. Run the corrected preparation notebook.")
    data_root = root/"official_data"/"Phase_4_CJPE"
    identity = read_json(data_root/"download_manifest.json")
    if identity["sha256"] != index["identity"]["official_data_sha256"]:
        raise ValueError("Prepared data and cache identities disagree.")
    for suffix, expected in identity["sha256"].items():
        if sha256(data_root/suffix) != expected:
            raise ValueError("Prepared official JSON changed: " + suffix)
    documents, seen = [], set()
    for row in index["cases"]:
        key = row["row_uid"]
        if key in seen:
            raise ValueError("Duplicate prepared case identity.")
        seen.add(key)
        path = cache/row["token_file"]
        if sha256(path) != index["cache_sha256"][row["token_file"]]:
            raise ValueError("Prepared token-cache bytes changed.")
        with np.load(path, allow_pickle=False) as z:
            ids = z["ids"].copy()
        if ids.ndim != 1 or len(ids) != row["content_tokens"]:
            raise ValueError("Prepared token-cache length mismatch.")
        documents.append(dict(row, token_ids=ids))
    return documents, {"data": identity, "tokenizer_sha256": index["identity"]["tokenizer_sha256"],
                       "cache_fingerprint": index["fingerprint"]}

Overwriting /kaggle/working/cs781_phase4_prep_src/cjpe_reuse.py


In [12]:
%%writefile /kaggle/working/cs781_phase4_prep_src/cjpe_workflow.py
"""Preparation workflow and compact diagnostics; real training is intentionally absent."""
import gc
import importlib.metadata
import json
import platform
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path
from cjpe_data import (sha256, write_json, read_json, safe_error, discover, official_data,
    load_documents, load_tokenizer, tokenize_documents, audit_documents, audit_chunk_policies,
    audit_record_ids, evaluation_id)


ASSETS = {
    "decoder": {"dataset": "decoder-004", "hint": "decoder_004_mixed217m",
        "sha256": "fd5add7ccc1d519a6f547fbd9a5c86a53276db8e09fb93c165d96531685d0bb7",
        "source": "phase1_decoder.py",
        "source_sha256": "c40066ffae9e41ee87d884450e0be64170eb5be49b682e3041796ddca6574a96",
        "method": "upper_layers_and_final_norm"},
    "encoder_decoder": {"dataset": "encdec-2", "hint": "encdec_002_mixed217m_stable",
        "sha256": "2a2d34a8df48bce7b6ab8132ea011803e2d044df8eaacde110046653b85516a5",
        "source": "architecture_models.py",
        "source_sha256": "24f762e290b90a47ed54bd7314950ef2883d77b56d6f4fce0a02f39bf24a1e04",
        "method": "bottleneck_adapters_and_layer_norm"},
    "prefix": {"dataset": "prefix-001", "hint": "prefix_001_mixed217m",
        "sha256": "5308feee0a0df36b6609979002e5709de1ad1ecf647923be2e07752b1481ddd0",
        "source": "architecture_models.py",
        "source_sha256": "24f762e290b90a47ed54bd7314950ef2883d77b56d6f4fce0a02f39bf24a1e04",
        "method": "output_side_lorma_plus"},
    "efficient": {"dataset": "efficient-002", "hint": "efficient",
        "sha256": "7117b452f4ba76bf8590a633c6b2bfe7472a328aefed7127de171d73b68bf3ea",
        "source": "architecture_models.py",
        "source_sha256": "4f2db692383361b59ca1e606ef8cddebb9d1d64fa7ace806b7d5766a735c9937",
        "method": "compact_student_gold_and_distillation"},
}
TOKENIZER_SHA = "080c6d2ce49a5c9ccfdd707145ec2b49c8a3f3d2e676e30d9c6e93c948900d9c"


def resolve_assets(config, root, src):
    result = {}
    for architecture, specification in ASSETS.items():
        print("Checking original Phase 1 input:", specification["dataset"], flush=True)
        try:
            path = discover("best_model.pt", config["input_root"],
                config["phase1_inputs"].get(architecture), [specification["hint"], specification["dataset"]],
                Path(root)/"extracted_inputs")
            actual = sha256(path)
            if actual != specification["sha256"]:
                raise ValueError("Checkpoint bytes differ from the original selected Phase 1 checkpoint. Expected " +
                                 specification["sha256"] + "; found " + actual)
            native = path.parent/"sources"/specification["source"]
            if native.exists() and sha256(native) != specification["source_sha256"]:
                raise ValueError("The checkpoint's native source differs from its reviewed Phase 1 implementation.")
            embedded = Path(src)/("phase1_decoder.py" if architecture == "decoder" else
                                  "phase1_efficient.py" if architecture == "efficient" else "phase1_shared.py")
            if sha256(embedded) != specification["source_sha256"]:
                raise ValueError("The embedded Phase 1 implementation was changed.")
            result[architecture] = dict(specification, status="passed", path=str(path),
                bytes=path.stat().st_size, native_source_present=native.exists(), embedded_original_source_verified=True)
        except Exception as exc:
            result[architecture] = dict(specification, status="failed", error=safe_error(exc))
    return result


def evaluator_check(documents, reports, src):
    import sklearn.metrics as metrics
    import zipfile
    reports = Path(reports)
    folder = reports/"format_check_only"
    folder.mkdir(exist_ok=True)
    counts = {label: sum(d["gold"] == label for d in documents if d["split"] == "train") for label in (0, 1)}
    majority = max(counts, key=lambda label: (counts[label], -label))
    dev = [d for d in documents if d["split"] == "dev"]
    test = [d for d in documents if d["split"] == "test"]
    truth = [{"id": evaluation_id(d), "gold": int(d["gold"])} for d in dev]
    predictions = [{"id": evaluation_id(d), "pred": int(majority)} for d in dev]
    write_json(folder/"dev_id_mapping.json", [
        {"local_evaluation_id": evaluation_id(d), "original_case_id": d["id"],
         "row_uid": d["row_uid"], "row_index": d["row_index"]} for d in dev])
    write_json(folder/"dev_truth.json", truth)
    write_json(folder/"dev_constant_prediction.json", predictions)
    result = subprocess.run([sys.executable, str(Path(src)/"evaluate_phase4.py"), "--truth",
        str(folder/"dev_truth.json"), "--prediction", str(folder/"dev_constant_prediction.json")],
        check=True, text=True, capture_output=True, timeout=120)
    expected = metrics.f1_score([d["gold"] for d in dev], [majority]*len(dev),
                               average="macro", labels=[0, 1], zero_division=0)
    scored = next(float(line.split(":")[1]) for line in result.stdout.splitlines() if line.startswith("Macro-F1:"))
    if abs(scored-expected) > 1e-6:
        raise AssertionError("Official evaluator and local macro-F1 disagree.")
    example = [{"id": d["id"], "pred": int(majority)} for d in test]
    payload = json.dumps(example, ensure_ascii=False).encode()
    archive = folder/"FORMAT_CHECK_ONLY_DO_NOT_SUBMIT.zip"
    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as z:
        z.writestr("prediction.json", payload)
    with zipfile.ZipFile(archive) as z:
        if z.namelist() != ["prediction.json"]:
            raise AssertionError("Prediction JSON must be at the ZIP root.")
        restored = json.loads(z.read("prediction.json"))
        if [r["id"] for r in restored] != [d["id"] for d in test]:
            raise AssertionError("Test IDs or order changed.")
        if any(type(r["pred"]) is not int or r["pred"] not in (0, 1) for r in restored):
            raise AssertionError("Invalid binary export type.")
    (folder/"DO_NOT_SUBMIT.txt").write_text(
        "This is a constant-label FORMAT CHECK ONLY. It is not a trained-model submission.\n")
    return {"status": "passed", "official_evaluator_output": result.stdout, "test_cases": len(test),
            "local_development_ids": "unique row_uid; original IDs retained in dev_id_mapping.json",
            "exact_test_ids_preserved": True, "trained_submission_generated": False}


def save_plots(documents, reports):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for split in ("train", "dev", "test"):
        values = sorted(len(d["token_ids"]) for d in documents if d["split"] == split)
        axes[0].plot(values, [(i+1)/len(values) for i in range(len(values))], label=split)
    axes[0].set(xscale="log", xlabel="Case content tokens", ylabel="Empirical cumulative proportion")
    axes[0].legend()
    counts = [[sum(d["split"] == split and d["gold"] == label for d in documents) for label in (0, 1)]
              for split in ("train", "dev")]
    x = range(2)
    axes[1].bar([i-.18 for i in x], counts[0], width=.36, label="train")
    axes[1].bar([i+.18 for i in x], counts[1], width=.36, label="dev")
    axes[1].set(xticks=[0, 1], xticklabels=["0", "1"], ylabel="Cases", xlabel="Official numeric outcome")
    axes[1].legend()
    fig.tight_layout()
    fig.savefig(Path(reports)/"data_overview.png", dpi=140)
    plt.close(fig)


def export_bundles(root, src, config, include_data):
    root, src = Path(root), Path(src)
    destination = Path(config["output_root"])
    destination.mkdir(parents=True, exist_ok=True)
    diagnostics = destination/(config["run_id"]+"_diagnostics.zip")
    prepared = destination/"phase4_cjpe_prepared_data.zip"
    with zipfile.ZipFile(str(diagnostics)+".tmp", "w", zipfile.ZIP_DEFLATED) as z:
        for directory, prefix in ((root/"reports", "reports"), (src, "sources")):
            for path in sorted(directory.rglob("*")):
                if path.is_file() and path.suffix in (".py", ".json", ".csv", ".png", ".md", ".txt", ".zip"):
                    z.write(path, config["run_id"]+"/"+prefix+"/"+str(path.relative_to(directory)))
    Path(str(diagnostics)+".tmp").replace(diagnostics)
    if include_data:
        with zipfile.ZipFile(str(prepared)+".tmp", "w", zipfile.ZIP_DEFLATED, compresslevel=3) as z:
            data = root/"official_data"/"Phase_4_CJPE"
            for suffix in ("Train/train.json", "Train/dev.json", "Test/test.json", "download_manifest.json"):
                z.write(data/suffix, "phase4_cjpe_prepared/official_data/Phase_4_CJPE/"+suffix)
            for p in sorted((root/"token_cache").glob("*")):
                if p.suffix in (".npz", ".json"):
                    z.write(p, "phase4_cjpe_prepared/token_cache/"+p.name)
            for directory, prefix in ((root/"reports", "reports"), (src, "sources")):
                for path in sorted(directory.rglob("*")):
                    if path.is_file() and path.suffix in (".py", ".json", ".csv", ".md", ".txt"):
                        z.write(path, "phase4_cjpe_prepared/"+prefix+"/"+str(path.relative_to(directory)))
        Path(str(prepared)+".tmp").replace(prepared)
    return {"diagnostics": str(diagnostics), "prepared_data": str(prepared) if include_data else None}


def run_preparation(config, source_dir):
    root, src = Path(config["working_root"]), Path(source_dir)
    reports = root/"reports"
    if reports.exists() and any(reports.iterdir()):
        previous = root/"previous_reports"/time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
        previous.parent.mkdir(parents=True, exist_ok=True)
        suffix = 1
        while previous.exists():
            previous = previous.with_name(previous.name+"_"+str(suffix))
            suffix += 1
        shutil.move(str(reports), str(previous))
    reports.mkdir(parents=True, exist_ok=True)
    summary = {"status": "running", "run_id": config["run_id"], "full_training_run": False,
               "minimum_required_submissions": 4, "models": {}, "started_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())}
    prepared = False
    try:
        import torch
        from cjpe_checks import helper_checks, architecture_probe
        torch.set_num_threads(2)
        environment = {"python": sys.version, "platform": platform.platform(),
            "torch": torch.__version__, "cuda_runtime": torch.version.cuda,
            "gpu_count": torch.cuda.device_count(), "gpus": [
                {"device": i, "name": torch.cuda.get_device_name(i),
                 "total_gib": torch.cuda.get_device_properties(i).total_memory/2**30}
                for i in range(torch.cuda.device_count())],
            "working_free_gib": shutil.disk_usage(root).free/2**30}
        for name in ("numpy", "pandas", "scikit-learn", "huggingface_hub", "matplotlib"):
            try:
                environment[name] = importlib.metadata.version(name)
            except importlib.metadata.PackageNotFoundError:
                environment[name] = None
        write_json(reports/"environment.json", environment)
        write_json(reports/"configuration.json", config)
        write_json(reports/"source_manifest.json", {p.name: sha256(p) for p in sorted(src.glob("*.py"))})
        print("GPUs:", environment["gpus"], flush=True)
        summary["helper_checks"] = helper_checks()
        assets = resolve_assets(config, root, src)
        write_json(reports/"phase1_inputs.json", assets)
        summary["input_status"] = {k: v["status"] for k, v in assets.items()}
        tokenizer_path = discover("tokenizer_state.pkl", config["input_root"], config["prep_input"],
            ["prep_002_mixed16k", "prep-2"], root/"extracted_inputs")
        tokenizer, signature, prep_manifest = load_tokenizer(tokenizer_path, TOKENIZER_SHA)
        tokenizer_report = {"path": str(tokenizer_path), "sha256": sha256(tokenizer_path),
            "signature": signature, "vocab_size": len(tokenizer.vocab),
            "reserved_ids": {"pad": 0, "unknown": 1, "bos": 2, "eos": 3},
            "round_trip_checks": "passed", "retrained": False}
        write_json(reports/"tokenizer.json", tokenizer_report)
        data, identity = official_data(root/"official_data"/"Phase_4_CJPE",
                                      config["hf_revision"], config["cjpe_data_input"])
        write_json(reports/"official_data_identity.json", identity)
        documents, schemas = load_documents(data, config["label_field"])
        id_audit = audit_record_ids(documents, reports)
        summary["record_id_audit"] = {k: v for k, v in id_audit.items() if k != "splits"}
        summary["record_id_audit"]["split_counts"] = {
            split: {k: v for k, v in info.items() if k != "collision_groups"}
            for split, info in id_audit["splits"].items()}
        print("Official row/ID counts:", summary["record_id_audit"]["split_counts"], flush=True)
        write_json(reports/"summary.json", summary)
        export_bundles(root, src, config, include_data=False)
        if id_audit["submission_blockers"]:
            raise ValueError(" ".join(id_audit["submission_blockers"]))
        summary["tokenization"] = tokenize_documents(documents, tokenizer, root/"token_cache",
            {"official_data_sha256": identity["sha256"], "tokenizer_sha256": tokenizer_report["sha256"],
             "text_list_policy": "newline_join", "normalization": "preserve supplied text",
             "record_identity_version": 2})
        prepared = True
        summary["data"] = audit_documents(documents, schemas, reports)
        summary["chunk_policy_rows"] = audit_chunk_policies(documents, reports, config["chunk_policy_candidates"])
        summary["evaluator_and_export"] = evaluator_check(documents, reports, src)
        save_plots(documents, reports)
        # A portable index references each atomic cache; raw texts remain in official JSON.
        index = read_json(root/"token_cache"/"index.json")
        index["cache_sha256"] = {row["token_file"]: sha256(root/"token_cache"/row["token_file"]) for row in index["cases"]}
        write_json(root/"token_cache"/"index.json", index)
        write_json(reports/"summary.json", summary)
        # Keep a download-ready handoff before GPU probes, even if a session later ends.
        export_bundles(root, src, config, include_data=True)
        devices_ok = torch.cuda.is_available() and (torch.cuda.device_count() >= 2 or
                                                    not config["settings"]["require_two_gpus"])
        for architecture in ("decoder", "encoder_decoder", "prefix", "efficient"):
            print("\nARCHITECTURE:", architecture, flush=True)
            if assets[architecture]["status"] != "passed":
                result = {"status": "failed", "error": assets[architecture]["error"]}
            elif not devices_ok:
                result = {"status": "not_run", "error": "Select T4 x2; preparation data are still retained."}
            elif not config["run_model_probes"]:
                result = {"status": "not_run", "error": "RUN_MODEL_PROBES is disabled."}
            else:
                result = architecture_probe(architecture, assets[architecture]["path"], signature,
                    config["settings"], documents, reports/architecture, torch.device("cuda:0"))
            summary["models"][architecture] = result
            write_json(reports/"summary.json", summary)
            export_bundles(root, src, config, include_data=False)
            print(architecture, ":", result["status"], flush=True)
            gc.collect()
        recipes = {}
        passing_geometry = []
        for model_report in summary["models"].values():
            passing_geometry.append({(r["total_tokens"], r["max_chunks"])
                for r in model_report.get("resources", []) if r["status"] == "passed"})
        common = set.intersection(*passing_geometry) if passing_geometry and all(passing_geometry) else set()
        geometry = max(common) if common else None
        summary["common_document_geometry"] = {"total_tokens": geometry[0], "max_chunks": geometry[1],
            "overlap": 64, "policy": "uniform"} if geometry else None
        for architecture, spec in ASSETS.items():
            probe = summary["models"].get(architecture, {})
            resource = probe.get("recommended_resource_configuration") or {}
            if geometry:
                matching = [r for r in probe.get("resources", []) if r["status"] == "passed"
                            and (r["total_tokens"], r["max_chunks"]) == geometry]
                resource = min(matching, key=lambda r: r["seconds_per_two_cases"])
            recipes[architecture] = {"method": spec["method"], "original_checkpoint_sha256": spec["sha256"],
                "input_dataset": spec["dataset"], "discovery_hint": spec["hint"],
                "settings": dict(config["settings"], chunk_microbatch=resource.get("chunk_microbatch", 1)),
                "document_selection": {"total_tokens": resource.get("total_tokens", 512),
                    "overlap": 64, "max_chunks": resource.get("max_chunks", 8), "policy": "uniform"},
                "global_cases_per_microbatch": 2, "proposed_accumulation_steps": 4,
                "resource_check_status": probe.get("status"), "learning_schedule": "Choose after reviewing measured case counts/runtime.",
                "task_head": "content mean + last/query; two chunk-position features; projection; manual BiGRU; attention; two logits",
                "real_teacher_targets_required": architecture == "efficient",
                "train_and_teacher_cache_join_key": "row_uid; never raw case ID alone",
                "development_evaluation_id": "row_uid",
                "test_submission_id": "unchanged official id",
                "student_gold_only_control_required": architecture == "efficient"}
        write_json(reports/"training_recipes.json", recipes)
        blockers = list(summary["data"]["training_blockers"])
        blockers += [k+": "+v["status"] for k, v in summary["models"].items() if v["status"] != "passed"]
        summary["training_blockers"] = blockers
        summary["status"] = "passed" if not blockers else "needs_review"
        summary["preparation_complete"] = prepared
        summary["remaining_work"] = ["Select final schedules and any needed configuration changes from diagnostics.",
            "Train three teacher candidates and select a CJPE teacher using development macro-F1.",
            "Cache genuine trained-teacher logits for training cases.",
            "Train matched gold-only and KD compact students.",
            "Evaluate selected checkpoints and export four trained-model submissions."]
        (reports/"README.md").write_text(
            "# Phase 4 preparation\n\n"
            "Read summary.json first. This archive contains no trained CJPE model.\n"
            "Temporary smoke/probe models were discarded; original Phase 1 files were never overwritten.\n"
            "Any synthetic teacher targets were wiring tests only, not a real teacher cache.\n"
            "Original numeric labels and test IDs are preserved. No ACCEPT/REJECT mapping was assumed.\n"
            "Repeated train/dev IDs are retained as distinct supplied rows with unique row_uid values.\n"
            "Use row_uid for training/KD alignment and local dev evaluation; raw id for test submission.\n"
            "Chunk-policy coverage is measured, not described as full-document coverage when capped.\n"
            "Runtime projections use maximum-shape stress probes, not a measured complete epoch.\n"
            "Keep the separate prepared-data ZIP and the five original Kaggle inputs for later training.\n")
    except KeyboardInterrupt:
        summary.update(status="interrupted", preparation_complete=prepared,
                       error="User interrupt; partial diagnostics and any finished token caches are retained.",
                       training_blockers=["Preparation/probes were interrupted."])
        print("Interrupted; exporting available diagnostics.", flush=True)
    except Exception as exc:
        summary.update(status="failed", preparation_complete=prepared, error_type=type(exc).__name__,
                       error=safe_error(exc), training_blockers=["Preparation did not complete; inspect this error."])
        print("PREPARATION ERROR:", safe_error(exc), flush=True)
    finally:
        summary["finished_utc"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
        write_json(reports/"summary.json", summary)
        paths = export_bundles(root, src, config, include_data=prepared)
        print("\nFINAL STATUS:", summary["status"], flush=True)
        print("SEND BACK:", paths["diagnostics"], flush=True)
        if paths["prepared_data"]:
            print("KEEP FOR TRAINING:", paths["prepared_data"], flush=True)
    return summary, paths

Overwriting /kaggle/working/cs781_phase4_prep_src/cjpe_workflow.py


In [13]:
%%writefile /kaggle/working/cs781_phase4_prep_src/evaluate_phase4.py
#!/usr/bin/env python3
"""Local evaluator for Phase 4.

Truth JSON: a list of objects, [{"id":"...", "gold":0|1}, ...]
Prediction JSON: [{"id":"...", "pred":0|1}, ...]
"""
import argparse,json,sys
from sklearn.metrics import f1_score,precision_score,recall_score

def read_truth(path):
    try:
        x=json.load(open(path,encoding="utf-8"))
    except json.JSONDecodeError as e:
        raise ValueError(f"Truth must be a JSON file containing a list of records: {e}")
    if not isinstance(x,list) or not x: raise ValueError('Truth must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'gold' not in r: raise ValueError(f'Truth item {n} must contain id and gold.')
        if not isinstance(r['id'],str) or not r['id'].strip() or isinstance(r['gold'],bool) or r['gold'] not in (0,1): raise ValueError(f'Invalid truth item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate truth ID: {r["id"]}')
        out[r['id']]=r['gold']
    return out

def read_pred(path):
    x=json.load(open(path,encoding='utf-8'))
    if not isinstance(x,list) or not x: raise ValueError('Prediction must be a non-empty JSON list.')
    out={}
    for n,r in enumerate(x):
        if not isinstance(r,dict) or 'id' not in r or 'pred' not in r: raise ValueError(f'Prediction item {n} must contain id and pred.')
        if not isinstance(r['id'],str) or isinstance(r['pred'],bool) or r['pred'] not in (0,1): raise ValueError(f'Invalid prediction item {n}.')
        if r['id'] in out: raise ValueError(f'Duplicate prediction ID: {r["id"]}')
        out[r['id']]=r['pred']
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 4 binary predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); a=ap.parse_args()
    t=read_truth(a.truth); p=read_pred(a.prediction)
    if set(t)!=set(p): raise ValueError(f"ID mismatch: missing={len(set(t)-set(p))}, extra={len(set(p)-set(t))}")
    yt=[t[i] for i in t]; yp=[p[i] for i in t]
    print(f'Examples: {len(t)}')
    print(f'Macro-F1: {f1_score(yt,yp,average="macro",labels=[0,1],zero_division=0):.6f}')
    print(f'Precision: {precision_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
    print(f'Recall: {recall_score(yt,yp,average="binary",pos_label=1,zero_division=0):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)

Overwriting /kaggle/working/cs781_phase4_prep_src/evaluate_phase4.py


## 4. Run the preflight

Each backbone receives 12 temporary optimizer updates, a checkpoint
round trip and resource probes over candidate chunk layouts. The
displayed epoch estimates extrapolate from maximum-shape stress batches.
The format check uses constant predictions to validate the submission schema.

In [14]:
import re
import zipfile

summary = None
paths = {}
sys.path.insert(0, str(SRC))
try:
    if DEPENDENCY_ERROR:
        raise RuntimeError(DEPENDENCY_ERROR)
    from cjpe_workflow import run_preparation
    summary, paths = run_preparation(CONFIG, SRC)
except (Exception, KeyboardInterrupt) as exc:
    error = re.sub(r"hf_[A-Za-z0-9]+", "[redacted]", str(exc))[:3000]
    folder = ROOT / "reports"
    folder.mkdir(exist_ok=True)
    summary = {"status": "failed", "error": error, "full_training_run": False}
    (folder / "bootstrap_failure.json").write_text(json.dumps(summary, indent=2))
    diagnostics = Path("/kaggle/working") / (RUN_ID + "_diagnostics.zip")
    with zipfile.ZipFile(diagnostics, "w", zipfile.ZIP_DEFLATED) as archive:
        for directory in (folder, SRC):
            for path in directory.rglob("*"):
                if path.is_file() and path.suffix in (".json", ".py", ".csv", ".txt", ".md"):
                    archive.write(path, RUN_ID + "/" + directory.name + "/" + str(path.relative_to(directory)))
    paths = {"diagnostics": str(diagnostics), "prepared_data": None}
    print("Setup failed:", error)
    print("Diagnostics:", diagnostics)

GPUs: [{'device': 0, 'name': 'Tesla T4', 'total_gib': 14.56219482421875}, {'device': 1, 'name': 'Tesla T4', 'total_gib': 14.56219482421875}]
Checking original Phase 1 input: decoder-004
Checking original Phase 1 input: encdec-2
Checking original Phase 1 input: prefix-001
Checking original Phase 1 input: efficient-002
Reused verified official CJPE files at commit 93b0059be756c6c461f3a225de25aabda7d5f84a
Official row/ID counts: {'train': {'provided_rows': 37387, 'unique_original_case_ids': 32305, 'repeated_original_id_groups': 5082, 'extra_rows_with_repeated_ids': 5082}, 'dev': {'provided_rows': 1988, 'unique_original_case_ids': 994, 'repeated_original_id_groups': 994, 'extra_rows_with_repeated_ids': 994}, 'test': {'provided_rows': 1517, 'unique_original_case_ids': 1517, 'repeated_original_id_groups': 0, 'extra_rows_with_repeated_ids': 0}}
Tokenized/reused 1 / 40892 cases | 0.0 minutes
Tokenized/reused 3147 / 40892 cases | 0.4 minutes
Tokenized/reused 7509 / 40892 cases | 0.8 minutes
Tok

## 5. Review the preparation

The outputs show row counts, document lengths, parameter counts and the
probe status for each model. Mount `phase4_cjpe_prepared_data.zip` in the
training notebooks. The training loader can pack it into
`phase4_cjpe_fast_cache.zip` for subsequent sessions. The diagnostics ZIP
records the data audit and preflight reports.

In [15]:
from IPython.display import display, FileLink, Markdown

display(Markdown("**Preparation status: " + str(summary.get("status", "unknown")) + "**"))
if summary.get("data"):
    import pandas as pd
    display(pd.DataFrame([
        {"split": split, "provided_rows": summary["data"][split]["cases"],
         "unique_original_case_ids": summary["data"][split]["unique_original_case_ids"],
         "label_counts": summary["data"][split]["labels"],
         "median_tokens": summary["data"][split]["content_tokens"]["50"],
         "p95_tokens": summary["data"][split]["content_tokens"]["95"]}
        for split in ("train", "dev", "test")
    ]))
if summary.get("models"):
    import pandas as pd
    display(pd.DataFrame([
        {"architecture": name, "status": report["status"],
         "total_parameters": report.get("parameters", {}).get("total_parameters"),
         "trainable_parameters": report.get("parameters", {}).get("trainable_parameters"),
         "stress_epoch_minutes": report.get("runtime_projection", {}).get("stress_based_train_epoch_minutes")}
        for name, report in summary["models"].items()
    ]))
if paths.get("diagnostics"):
    display(Markdown("**Preparation diagnostics**"))
    display(FileLink(paths["diagnostics"]))
if paths.get("prepared_data"):
    display(Markdown("**Prepared data for training**"))
    display(FileLink(paths["prepared_data"]))

**Preparation status: passed**

,split,provided_rows,unique_original_case_ids,label_counts,median_tokens,p95_tokens
0,train,37387,32305,"{'0': 22067, '1': 15320}",4924.0,20934.70
1,dev,1988,994,"{'0': 994, '1': 994}",6436.0,21292.05
2,test,1517,1517,{},6695.0,22503.20


,architecture,status,total_parameters,trainable_parameters,stress_epoch_minutes
0,decoder,passed,218698626,51209090,139.961328
1,encoder_decoder,passed,223187970,5405698,308.947442
2,prefix,passed,223419266,5557122,650.827443
3,efficient,passed,55003266,55003266,101.158843


**Preparation diagnostics**

/kaggle/working/phase4_cjpe_preparation_001_diagnostics.zip

**Prepared data for training**

/kaggle/working/phase4_cjpe_prepared_data.zip

## Method references

- Long-document judgment prediction. [ILDC for CJPE](https://aclanthology.org/2021.acl-long.313/)
- Bottleneck adapters. [Parameter-Efficient Transfer Learning for NLP](https://proceedings.mlr.press/v97/houlsby19a.html)
- Output-side multiplicative low-rank adaptation. [LoRMA](https://aclanthology.org/2025.findings-acl.527/)
- Temperature-scaled soft targets. [Distilling the Knowledge in a Neural Network](https://arxiv.org/abs/1503.02531)